<a href="https://colab.research.google.com/github/warry258/colab/blob/main/TXT_to_IMAGE_GGUF_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 💾 挂载Google硬盘
%%capture
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 🗑️ 控制显存碎片配置
import os, sys

# 必须在 import torch / diffusers / transformers 之前运行
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# 防呆：如果 torch 已经被导入了，就说明这格跑晚了（本次不会生效）
if "torch" in sys.modules:
    raise RuntimeError("torch 已导入：allocator 配置本次不会生效。请重启 runtime 后第一格先跑这个 cell。")

print("✅ PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

In [ ]:
# @title 🎛️ 环境配置
# ====================================
# 合并版：包含基础环境、插件、模型下载与验证
# 修复：增加 ESRGAN 模型（像素空间放大）
# 修复：ComfyUI-GGUF 自引用问题（先注册 sys.modules 再 exec_module）
# ====================================

import subprocess, sys, os, torch, psutil

# ── 基础参数 ──────────────────────────────────────────
COMFYUI_PATH = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFYUI_PATH}/custom_nodes'

# ── Step 1: 克隆并安装 ComfyUI ─────────────────────────
print("📦 Step 1: 安装 ComfyUI 基础环境...")
if not os.path.exists(COMFYUI_PATH):
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/comfyanonymous/ComfyUI', COMFYUI_PATH],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("   ✓ ComfyUI 克隆完成")
else:
    print("   ✓ ComfyUI 已存在，跳过")

%cd -q {COMFYUI_PATH}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location',
                '-r', 'requirements.txt'],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("   ✓ ComfyUI 依赖安装完成")

# ── Step 2: 安装 GGUF 插件 ─────────────────────────────
print("\n📦 Step 2: 安装 ComfyUI-GGUF 插件...")
gguf_path = f'{CUSTOM_NODES}/ComfyUI-GGUF'
if not os.path.exists(gguf_path):
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/city96/ComfyUI-GGUF', gguf_path],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("   ✓ ComfyUI-GGUF 克隆完成")
else:
    print("   ✓ ComfyUI-GGUF 已存在，跳过")

for req_file in [f'{gguf_path}/requirements.txt']:
    if os.path.exists(req_file):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location',
                        '-r', req_file],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location', 'gguf'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("   ✓ GGUF 依赖安装完成")

# ── Step 3: 安装高阶节点与放大模型 ──────────────────────
print("\n📦 Step 3: 下载插件与模型...")

# 3.1 Ultimate SD Upscale
usdu_path = f'{CUSTOM_NODES}/ComfyUI_UltimateSDUpscale'
if not os.path.exists(usdu_path):
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/ssitu/ComfyUI_UltimateSDUpscale', usdu_path],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("   ✓ Ultimate SD Upscale 插件已下载")
else:
    print("   ✓ Ultimate SD Upscale 已存在，跳过")

# 3.2 SeedVarianceEnhancer
sv_path = f'{CUSTOM_NODES}/SeedVarianceEnhancer'
if not os.path.exists(sv_path):
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/ChangeTheConstants/SeedVarianceEnhancer', sv_path],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("   ✓ SeedVarianceEnhancer 插件已下载")
else:
    print("   ✓ SeedVarianceEnhancer 已存在，跳过")

# 3.3 下载 4x-UltraSharp 放大模型
upscale_dir = f'{COMFYUI_PATH}/models/upscale_models'
os.makedirs(upscale_dir, exist_ok=True)

ultrasharp_path = f'{upscale_dir}/4x-UltraSharp.pth'
if not os.path.exists(ultrasharp_path):
    subprocess.run(['wget', '-q', '-nc', '-O', ultrasharp_path,
                    'https://huggingface.co/lokcx/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth'],
                   check=True)
    print("   ✓ 4x-UltraSharp 放大模型已下载 (67MB)")
else:
    print("   ✓ 4x-UltraSharp 已存在，跳过")

# 3.4 下载 Real-ESRGAN x4plus
esrgan_path = f'{upscale_dir}/RealESRGAN_x4plus.pth'
if not os.path.exists(esrgan_path):
    print("   ⏳ 下载 Real-ESRGAN x4plus 模型 (64MB)...")
    subprocess.run(['wget', '-q', '-nc', '-O', esrgan_path,
                    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'],
                   check=True)
    print("   ✓ Real-ESRGAN x4plus 已下载 (64MB)")
else:
    print("   ✓ Real-ESRGAN x4plus 已存在，跳过")

# ── Step 4: 安装辅助工具 ────────────────────────────────
print("\n📦 Step 4: 安装辅助工具...")
for pkg in ['psutil', 'ipywidgets', 'huggingface_hub']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location', pkg],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("   ✓ 辅助工具安装完成")

# ── Step 5: SDPA 后端配置 ───────────────────────────────
print("\n⚙️  Step 5: 配置 PyTorch SDPA 后端...")
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
print(f"   • flash_sdp        : {torch.backends.cuda.flash_sdp_enabled()}")
print(f"   • mem_efficient_sdp: {torch.backends.cuda.mem_efficient_sdp_enabled()} ✅")
print(f"   • math_sdp         : {torch.backends.cuda.math_sdp_enabled()}")

# ── Step 6: 加载插件节点注册 ────────────────────────────
print("\n🔌 Step 6: 加载插件节点...")
sys.path.insert(0, COMFYUI_PATH)
sys.path.insert(0, CUSTOM_NODES)
os.chdir(COMFYUI_PATH)

import importlib, importlib.util, glob as _glob
from nodes import NODE_CLASS_MAPPINGS

# 6.1 注册 ComfyUI-GGUF
# 修复：先将模块占位注册到 sys.modules，
#       再执行 exec_module，避免 __init__.py
#       内部 from ComfyUI_GGUF.xxx import yyy 自引用失败
try:
    _gguf_init = f'{gguf_path}/__init__.py'
    if os.path.exists(_gguf_init):
        _spec = importlib.util.spec_from_file_location("ComfyUI_GGUF", _gguf_init)
        _mod  = importlib.util.module_from_spec(_spec)
        sys.modules["ComfyUI_GGUF"] = _mod       # ← 先注册占位
        _spec.loader.exec_module(_mod)            # ← 再执行，自引用可解析
    else:
        # 无 __init__.py：将目录加入 sys.path 后逐文件扫描
        if gguf_path not in sys.path:
            sys.path.insert(0, gguf_path)
        _mod = None
        for _py in _glob.glob(f'{gguf_path}/*.py'):
            if os.path.basename(_py).startswith('_'):
                continue
            _name = os.path.splitext(os.path.basename(_py))[0]
            _spec = importlib.util.spec_from_file_location(_name, _py)
            _m    = importlib.util.module_from_spec(_spec)
            sys.modules[_name] = _m              # ← 同样先注册
            _spec.loader.exec_module(_m)
            if getattr(_m, 'NODE_CLASS_MAPPINGS', {}):
                _mod = _m; break

    _gguf_nodes = getattr(_mod, 'NODE_CLASS_MAPPINGS', {}) if _mod else {}
    NODE_CLASS_MAPPINGS.update(_gguf_nodes)
    print(f"   ✓ ComfyUI-GGUF 节点注册成功: "
          f"{list(_gguf_nodes.keys())[:4]}"
          f"{'...' if len(_gguf_nodes) > 4 else ''}")
except Exception as e:
    print(f"   ❌ ComfyUI-GGUF 节点加载失败: {e}")

# 6.2 注册 SeedVarianceEnhancer
try:
    _sve_mod   = importlib.import_module("SeedVarianceEnhancer")
    _sve_nodes = getattr(_sve_mod, 'NODE_CLASS_MAPPINGS', {})
    NODE_CLASS_MAPPINGS.update(_sve_nodes)
    print(f"   ✓ SeedVarianceEnhancer 节点注册成功: {list(_sve_nodes.keys())}")
except Exception as e:
    print(f"   ❌ SeedVarianceEnhancer 节点加载失败: {e}")

# ── Step 7: 环境验证 ────────────────────────────────────
print("\n" + "="*60)
print("✅ 环境验证报告")
print("="*60)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU: {device_name}")
    props = torch.cuda.get_device_properties(0)
    if hasattr(props, 'total_memory'):
        vram = props.total_memory / 1024**3
    elif hasattr(props, 'total_mem'):
        vram = props.total_mem / 1024**3
    else:
        vram = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"VRAM: {vram:.1f} GB")
else:
    print("⚠️  未检测到 GPU，将使用 CPU 运行（速度较慢）")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB (可用 {ram.available / 1024**3:.1f} GB)")

checks = [
    ("ComfyUI",              os.path.exists(f'{COMFYUI_PATH}/main.py')),
    ("GGUF 插件",             os.path.exists(gguf_path)),
    ("UltimateSDUpscale",    os.path.exists(usdu_path)),
    ("SeedVarianceEnhancer", os.path.exists(sv_path)),
    ("4x-UltraSharp",        os.path.exists(ultrasharp_path)),
    ("Real-ESRGAN x4plus",   os.path.exists(esrgan_path)),
    ("GGUF 节点已注册",       bool(_gguf_nodes)),
    ("SVE 节点已注册",        "SeedVarianceEnhancer" in NODE_CLASS_MAPPINGS),
]
print("\n📋 组件状态:")
for name, ok in checks:
    print(f"   {'✓' if ok else '✗'} {name}")

print("\n🎉 环境配置全部完成！可启动 ComfyUI 使用 GGUF 模型 + 高清放大（含 ESRGAN）")
print("="*60)

In [ ]:
# @title 🧩 ComfyUI LoRA 下载与资源管理器
import ipywidgets as widgets
from IPython.display import display
import os, requests
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

# ════════════════════════════════════════════════════════
# 1. 核心路径配置
# ════════════════════════════════════════════════════════
LORA_DIR = "/content/ComfyUI/models/loras"
os.makedirs(LORA_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════
# 2. 下载与管理工具
# ════════════════════════════════════════════════════════
def download_from_url(url, custom_filename=None, force=False):
    filename = custom_filename or os.path.basename(url.split("?")[0])
    if not filename.endswith((".safetensors", ".pt")): filename += ".safetensors"
    local_path = os.path.join(LORA_DIR, filename)
    if os.path.exists(local_path) and not force: return local_path, "already_exists"
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        return local_path, "downloaded"
    except Exception as e:
        return None, str(e)

def download_from_hub(repo_id, filename, force=False):
    local_path = os.path.join(LORA_DIR, os.path.basename(filename))
    if os.path.exists(local_path) and not force: return local_path, "already_exists"
    try:
        downloaded = hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=LORA_DIR, local_dir_use_symlinks=False, force_download=force,
        )
        return downloaded, "downloaded"
    except Exception as e:
        return None, str(e)

def list_repo_safetensors(repo_id):
    try: return[f for f in list_repo_files(repo_id) if f.endswith((".safetensors", ".pt"))]
    except Exception: return[]

# ════════════════════════════════════════════════════════
# 3. UI 构建
# ════════════════════════════════════════════════════════
output_widget = widgets.Output()

# ── 来源选择 ──
source_method = widgets.RadioButtons(
    options=[("📂 管理本地文件", "local"), ("📦 HuggingFace Repo 下载", "repo"), ("🔗 直链 URL 下载", "url")],
    value="repo",
    description="操作模式:",
    layout=widgets.Layout(width="400px")
)

# ── 本地管理区 ──
local_file_dropdown = widgets.Dropdown(options=[], layout=widgets.Layout(flex="1 1 auto"))
delete_btn = widgets.Button(description="🗑️ 删除选中", button_style="danger", layout=widgets.Layout(width="110px"))
local_file_row = widgets.HBox([widgets.Label("本地文件:", layout=widgets.Layout(width="70px")), local_file_dropdown, delete_btn], layout=widgets.Layout(width="100%", display="none"))

# ── URL 下载区 ──
url_input = widgets.Text(placeholder="https://.../model.safetensors", layout=widgets.Layout(flex="1 1 auto"))
url_row = widgets.HBox([widgets.Label("URL链接:", layout=widgets.Layout(width="70px")), url_input], layout=widgets.Layout(width="100%", display="none"))

# ── Repo 下载区 ──
repo_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("alibaba-pai/Z-Image-Fun-Lora-Distill", "alibaba-pai/Z-Image-Fun-Lora-Distill"),
        ("nphSi/Z-Image-Lora", "nphSi/Z-Image-Lora"),
        ("ifmylove2011/girlslike-zimage", "ifmylove2011/girlslike-zimage"),
    ],
    value="alibaba-pai/Z-Image-Fun-Lora-Distill",
    layout=widgets.Layout(width="250px")
)

repo_input = widgets.Text(
    value="alibaba-pai/Z-Image-Fun-Lora-Distill",
    layout=widgets.Layout(flex="1 1 auto")
)

def on_preset_change(change):
    if change['new']:
        repo_input.value = change['new']

repo_preset.observe(on_preset_change, names='value')

browse_btn = widgets.Button(description="📂 浏览 Repo", layout=widgets.Layout(width="110px"))
repo_row = widgets.HBox(
    [widgets.Label("Repo ID:", layout=widgets.Layout(width="70px")),
     repo_preset,
     repo_input,
     browse_btn],
    layout=widgets.Layout(width="100%")
)

filename_input = widgets.Text(placeholder="保存的文件名 (例如: my_lora.safetensors)", layout=widgets.Layout(flex="1 1 auto"))
filename_row = widgets.HBox([widgets.Label("文件名:", layout=widgets.Layout(width="70px")), filename_input], layout=widgets.Layout(width="100%"))

file_selector = widgets.Select(options=[], rows=6, layout=widgets.Layout(flex="1 1 auto", display="none"))
confirm_file_btn = widgets.Button(description="✅ 选定文件", button_style="success", layout=widgets.Layout(width="110px", display="none"))
file_select_row = widgets.HBox([widgets.Label("Repo文件:", layout=widgets.Layout(width="70px")), file_selector, confirm_file_btn], layout=widgets.Layout(width="100%"))

# ── 选项与操作 ──
force_check   = widgets.Checkbox(value=False, description="强制覆盖已存在文件", indent=False)
execute_btn = widgets.Button(description="🚀 执行下载", button_style="primary", layout=widgets.Layout(width="150px", height="38px"))
refresh_btn = widgets.Button(description="🔄 刷新本地", layout=widgets.Layout(width="110px", height="38px"))
status_html = widgets.HTML(value="<span style='color:gray;'>准备就绪</span>")
file_list_html = widgets.HTML(value="<i>暂无文件</i>")

# ── 动态 UI 事件 ──
def on_method_change(change):
    method = change["new"]
    local_file_row.layout.display  = "flex" if method == "local" else "none"
    url_row.layout.display         = "flex" if method == "url" else "none"
    repo_row.layout.display        = "flex" if method == "repo" else "none"
    filename_row.layout.display    = "flex" if method != "local" else "none"
    force_check.layout.display     = "flex" if method != "local" else "none"
    execute_btn.layout.display     = "block" if method != "local" else "none"
    file_selector.layout.display   = "none"
    confirm_file_btn.layout.display= "none"

source_method.observe(on_method_change, names="value")

def on_browse(b):
    repo = repo_input.value.strip()
    if not repo:
        with output_widget: output_widget.clear_output(); print("❌ 请填写 Repo ID")
        return
    with output_widget:
        output_widget.clear_output(); print(f"📡 获取 {repo} 文件列表...")
        files = list_repo_safetensors(repo)
        if not files: print("❌ 未找到 .safetensors 文件"); file_selector.options = []
        else:
            file_selector.options = files
            file_selector.value   = files[0]
            file_selector.layout.display    = "flex"
            confirm_file_btn.layout.display = "block"
            print(f"✅ 找到 {len(files)} 个文件，请在列表选中后点击【选定文件】")

browse_btn.on_click(on_browse)

def on_confirm_file(b):
    if file_selector.value:
        filename_input.value = file_selector.value
        file_selector.layout.display = "none"
        confirm_file_btn.layout.display = "none"
        with output_widget: output_widget.clear_output(); print(f"✅ 已锁定下载目标: {file_selector.value}")

confirm_file_btn.on_click(on_confirm_file)

def refresh_file_list():
    if not os.path.exists(LORA_DIR): return
    files =[f for f in os.listdir(LORA_DIR) if f.endswith((".safetensors", ".pt"))]
    local_file_dropdown.options = sorted(files)
    if files: local_file_dropdown.value = sorted(files)[0]

    if not files:
        file_list_html.value = "<i>Lora 文件夹为空</i>"
        return
    lines =[]
    for f in sorted(files):
        path = os.path.join(LORA_DIR, f)
        size = os.path.getsize(path) / 1024 / 1024
        lines.append(f"<div>🟢 {f}  <span style='color:gray;'>({size:.1f} MB)</span></div>")
    file_list_html.value = "<b>/models/loras/ 本地文件列表:</b><br>" + "".join(lines)

refresh_btn.on_click(lambda b: refresh_file_list())

def on_delete(b):
    target = local_file_dropdown.value
    if target:
        path = os.path.join(LORA_DIR, target)
        if os.path.exists(path):
            os.remove(path)
            with output_widget: output_widget.clear_output(); print(f"🗑️ 已删除: {target}")
            refresh_file_list()

delete_btn.on_click(on_delete)

# ── 核心执行逻辑 ──
def on_execute(b):
    output_widget.clear_output()
    execute_btn.disabled = True
    method = source_method.value
    force  = force_check.value

    def log(msg):
        with output_widget: print(msg)

    try:
        status_html.value = "<span style='color:orange;'>⏳ 下载中...请稍候</span>"

        if method == "url":
            url = url_input.value.strip()
            if not url: log("❌ 请输入 URL"); status_html.value = "❌"; return
            log(f"⬇️ 正在从直链下载...")
            local_path, result = download_from_url(url, filename_input.value.strip() or None, force)
        elif method == "repo":
            repo, fname = repo_input.value.strip(), filename_input.value.strip()
            if not repo or not fname: log("❌ 请填写 Repo ID 和文件名"); status_html.value = "❌"; return
            log(f"⬇️ 正在从 HuggingFace 极速拉取: {repo} -> {fname}")
            local_path, result = download_from_hub(repo, fname, force)

        if local_path:
            log(f"{'✅ 下载成功' if result == 'downloaded' else '⏭️ 文件已存在，跳过下载'}: {local_path}")
            log("💡 提示：去【出图面板】的 LoRA 槽位点击【🔄 刷新】即可使用！")
            status_html.value = "<span style='color:green;'>✅ 任务完成</span>"
        else:
            log(f"❌ 下载失败: {result}"); status_html.value = "<span style='color:red;'>❌ 失败</span>"

        refresh_file_list()

    except Exception as e:
        log(f"\n❌ 错误: {e}")
        status_html.value = "<span style='color:red;'>❌ 运行异常</span>"
    finally:
        execute_btn.disabled = False

execute_btn.on_click(on_execute)
refresh_file_list()

# ════════════════════════════════════════════════════════
# 4. 最终界面布局
# ════════════════════════════════════════════════════════
header = widgets.HTML("""
<div style='background:linear-gradient(135deg,#6366f1,#8b5cf6); color:white;padding:12px 16px;border-radius:8px;margin-bottom:8px;'>
  <h3 style='margin:0;'>📦 ComfyUI LoRA 下载与资源台</h3>
  <span style='font-size:12px;'>直接将模型拉取到 ComfyUI 目录，无需转换。下载后请在「出图面板」热加载生效。</span>
</div>
""")

ui = widgets.VBox([
    header,
    source_method,
    local_file_row,
    url_row,
    repo_row,
    file_select_row,
    filename_row,
    force_check,
    widgets.HBox([execute_btn, refresh_btn, status_html], layout=widgets.Layout(align_items='center', gap='10px')),
    widgets.HTML("<hr style='border-color:#334155; margin:10px 0;'>"),
    file_list_html,
    output_widget,
], layout=widgets.Layout(padding="16px", border="1px solid #334155", border_radius="12px", width="100%", max_width="900px", background="#0f172a"))

display(ui)

In [ ]:
# -*- coding: utf-8 -*-
# @title 🤖 模型管理器
# ============================================================
# 核心机制：
#   1. _deep_weld()          ── Trans/VAE 物理锁死显存，免疫 .to('cpu')
#   2. _destroy_te()         ── TE 真·销毁，彻底切断所有引用
#   3. _detach_cond()        ── embedding 与计算图断绝关系再传递
#   4. tiled_decode()        ── 2048×2048 分块 VAE 解码，峰值可控
#   5. vram_state 强制 HIGH_VRAM，封死 ComfyUI 自动 offload 入口
#
# 新增（串行按需模式）：
#   6. _install_gguf_mmap_tracker() ── 钩住 GGUFReader，追踪 mmap 对象
#   7. _flush_gguf_mmaps()          ── 加载后 MADV_DONTNEED 归还物理页
#   8. ZImageEngineSerial           ── Trans/TE 轮流上场，永不共存显存
#      流程：TE加载→编码→TE抛 ⟶ Trans流式加载→采样→Trans抛 ⟶ VAE解码(常驻)
# ============================================================

import ctypes                          # ← 新增
import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
print(f"flash_sdp        : {torch.backends.cuda.flash_sdp_enabled()}")
print(f"mem_efficient_sdp: {torch.backends.cuda.mem_efficient_sdp_enabled()}")

import ipywidgets as widgets
from IPython.display import display
import torch, os, gc, sys, time, random, shutil, ctypes
import psutil

try:
    import huggingface_hub
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])


# ══════════════════════════════════════════════════════════════
# 🔧 madvise 工具（新增）
# ══════════════════════════════════════════════════════════════
_MADV_DONTNEED = 4
_libc_mgr      = ctypes.CDLL("libc.so.6", use_errno=True)
_PAGE_MGR      = 4096

# 追踪每次 UnetLoaderGGUF 调用时内部打开的 np.memmap 对象
_gguf_mmap_registry = []


def _install_gguf_mmap_tracker():
    """
    在 gguf.GGUFReader.__init__ 上安装一次性 mmap 追踪钩子。
    加载完成后调用 _flush_gguf_mmaps() 即可 MADV_DONTNEED 整个文件映射，
    将 12.3GB 的物理页全部归还 OS，CPU RAM 立即回落。
    """
    try:
        import gguf as _gguf_lib
        if getattr(_gguf_lib.GGUFReader, '_mmap_tracked', False):
            return   # 已安装，幂等
        _orig_init = _gguf_lib.GGUFReader.__init__

        def _tracked_init(self, path, mode='r'):
            _orig_init(self, path, mode)
            data = getattr(self, '_data', None)
            if data is not None and hasattr(data, 'ctypes'):
                _gguf_mmap_registry.append(data)

        _gguf_lib.GGUFReader.__init__ = _tracked_init
        _gguf_lib.GGUFReader._mmap_tracked = True
        print("✅ GGUF mmap 追踪器已安装")
    except Exception as e:
        print(f"⚠️  GGUF mmap 追踪器安装失败（不影响运行）: {e}")


def _flush_gguf_mmaps():
    """
    MADV_DONTNEED 所有已追踪的 gguf mmap，归还物理页给 OS。
    调用时机：UnetLoaderGGUF.load_unet() 返回、deep_weld 完成之后立刻调用。
    返回归还的估算 GB 数。
    """
    freed_gb = 0.0
    for data in _gguf_mmap_registry:
        try:
            ptr    = data.ctypes.data
            nbytes = data.nbytes
            if not ptr or not nbytes:
                continue
            aligned = ptr & ~(_PAGE_MGR - 1)
            length  = (nbytes + (ptr - aligned) + _PAGE_MGR - 1) & ~(_PAGE_MGR - 1)
            ret = _libc_mgr.madvise(
                ctypes.c_void_p(aligned),
                ctypes.c_size_t(length),
                ctypes.c_int(_MADV_DONTNEED)
            )
            if ret == 0:
                freed_gb += nbytes / 1024 ** 3
        except Exception:
            pass
    _gguf_mmap_registry.clear()
    try:
        _libc_mgr.malloc_trim(0)
    except Exception:
        pass
    gc.collect()
    gc.collect()
    return freed_gb


# ══════════════════════════════════════════════════════════════
# 📂 路径常量
# ══════════════════════════════════════════════════════════════
COMFY_ROOT    = "/content/ComfyUI"
DIR_UNET      = os.path.join(COMFY_ROOT, "models", "unet")
DIR_DIFFUSION = os.path.join(COMFY_ROOT, "models", "diffusion_models")
DIR_CLIP      = os.path.join(COMFY_ROOT, "models", "clip")
DIR_VAE       = os.path.join(COMFY_ROOT, "models", "vae")
for d in [DIR_UNET, DIR_DIFFUSION, DIR_CLIP, DIR_VAE]:
    os.makedirs(d, exist_ok=True)

DEFAULT_REPO_UNET = "unsloth/Z-Image-Turbo-GGUF"
DEFAULT_REPO_TE   = "unsloth/Qwen3-4B-GGUF"
DEFAULT_REPO_VAE  = "Comfy-Org/z_image_turbo"
DEFAULT_VAE_PATH  = "split_files/vae/ae.safetensors"


# ══════════════════════════════════════════════════════════════
# 🔌 ComfyUI 运行时初始化
# ══════════════════════════════════════════════════════════════
_COMFY_INITIALIZED = False
_mm        = None
_csd       = None
_cu        = None
_fp        = None
_gguf_nodes = None


def _init_comfy(output_widget=None):
    global _COMFY_INITIALIZED, _mm, _csd, _cu, _fp, _gguf_nodes

    if _COMFY_INITIALIZED:
        return True

    def log(msg):
        if output_widget:
            with output_widget: print(msg)
        else:
            print(msg)

    gguf_old = os.path.join(COMFY_ROOT, "custom_nodes", "ComfyUI-GGUF")
    gguf_new = os.path.join(COMFY_ROOT, "custom_nodes", "ComfyUI_GGUF")
    if os.path.exists(gguf_old) and not os.path.exists(gguf_new):
        log("🔧 重命名 GGUF 目录: ComfyUI-GGUF → ComfyUI_GGUF")
        shutil.move(gguf_old, gguf_new)
    elif os.path.exists(gguf_new):
        log("✅ GGUF 目录已就绪")

    _orig_argv = sys.argv
    sys.argv = [sys.argv[0], '--gpu-only']

    for p in [COMFY_ROOT, os.path.join(COMFY_ROOT, "custom_nodes")]:
        if p not in sys.path:
            sys.path.insert(0, p)
    os.chdir(COMFY_ROOT)

    try:
        import comfy.model_management as mm_mod
        import comfy.sd as csd_mod
        import comfy.utils as cu_mod
        import folder_paths as fp_mod
        _mm, _csd, _cu, _fp = mm_mod, csd_mod, cu_mod, fp_mod
    except Exception as e:
        log(f"❌ ComfyUI 核心导入失败: {e}")
        sys.argv = _orig_argv
        return False
    finally:
        sys.argv = _orig_argv

    if hasattr(_fp, 'folder_names_and_paths'):
        for key, d in [
            ('unet', DIR_UNET),
            ('diffusion_models', DIR_UNET),
            ('clip', DIR_CLIP),
            ('vae', DIR_VAE),
        ]:
            if key in _fp.folder_names_and_paths:
                if d not in _fp.folder_names_and_paths[key][0]:
                    _fp.folder_names_and_paths[key][0].append(d)
            else:
                _fp.folder_names_and_paths[key] = (
                    [d], {'.gguf', '.safetensors', '.pt', '.pth'}
                )

    try:
        _mm.vram_state = _mm.VRAMState.HIGH_VRAM
        log("⚡ vram_state 锁定 → HIGH_VRAM")
    except Exception as e:
        log(f"⚠️  vram_state 锁定失败: {e}")

    try:
        from ComfyUI_GGUF import nodes as gn
        _gguf_nodes = gn
        log("✅ GGUF 节点注册成功")
    except Exception as e:
        log(f"❌ GGUF 节点加载失败: {e}")
        return False

    # ── 串行模式需要：安装 GGUF mmap 追踪器 ──────────────────
    _install_gguf_mmap_tracker()

    _COMFY_INITIALIZED = True
    return True


# ══════════════════════════════════════════════════════════════
# ⚙️  ZImageEngine（电焊模式，原版不变）
# ══════════════════════════════════════════════════════════════
class ZImageEngine:
    """
    Trans + VAE 物理锁死显存，TE 用后即抛。（原电焊模式）
    对外接口：
        engine.load(unet_name, te_name, vae_path)
        engine.generate(...)
        engine.vram_status()
        engine.destroy()
    """
    _MODE = "weld"   # 供外部识别引擎类型

    def __init__(self, output_widget=None):
        self.device  = None
        self.model   = None
        self.vae     = None
        self.clip    = None
        self._out    = output_widget

    def _log(self, msg):
        if self._out:
            with self._out: print(msg)
        else:
            print(msg)

    def _report(self, stage=""):
        if not torch.cuda.is_available(): return
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        ram   = psutil.virtual_memory()
        self._log(
            f"  📊 [{stage}]\n"
            f"     VRAM: {alloc:.2f}/{total:.1f} GB  (空闲 {total-alloc:.2f} GB)\n"
            f"     RAM:  {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f} GB"
        )

    def _deep_weld(self, name, obj):
        self._log(f"   [电焊] 切断 {name} 退回 RAM 的通道...")
        try:
            if   hasattr(obj, 'model'):             nn_model = obj.model
            elif hasattr(obj, 'first_stage_model'): nn_model = obj.first_stage_model
            else:                                    nn_model = obj

            nn_model.to(self.device)

            patcher = getattr(obj, 'patcher', obj)
            if hasattr(patcher, 'model_offload'):
                patcher.model_offload = lambda *a, **kw: None

            original_to = nn_model.to
            def fake_to(*args, **kwargs):
                target = args[0] if args else kwargs.get('device')
                if target is not None and 'cpu' in str(target).lower():
                    return nn_model
                return original_to(*args, **kwargs)
            nn_model.to = fake_to

            self._log(f"      ⚡ {name} 已永久免疫 .to('cpu')")
        except Exception as e:
            self._log(f"      ⚠️ 电焊异常: {e}")

    def load(self, unet_name, te_name, vae_path, output_widget=None):
        if output_widget:
            self._out = output_widget
        self.device = _mm.get_torch_device()

        self._log(f"\n{'='*60}")
        self._log("🚀 ZImageEngine 初始化（物理电焊防 Offload 版）")
        self._log(f"   设备: {self.device}")
        self._log(f"{'='*60}")
        self._report("初始状态")

        self._load_transformer(unet_name)
        self._load_vae(vae_path)

        self._default_te = te_name   # Cell 2 兼容

        self._log("\n✅ Trans/VAE 已物理锁定，TE 待命")
        self._report("初始化完成")

    def _load_transformer(self, unet_name):
        self._log(f"\n[常驻] 加载 Transformer: {unet_name}")
        t0 = time.time()
        loader = _gguf_nodes.UnetLoaderGGUF()
        (self.model,) = loader.load_unet(unet_name)
        self._deep_weld("Transformer", self.model)
        self._log(f"   ✅ Transformer 完成 ({time.time()-t0:.1f}s)")
        self._report("Transformer 焊死后")

    def _load_vae(self, vae_path):
        self._log(f"\n[常驻] 加载 VAE: {os.path.basename(vae_path)}")
        t0 = time.time()
        self.vae = _csd.VAE(sd=_cu.load_torch_file(vae_path))
        self._deep_weld("VAE", self.vae)
        self._log(f"   ✅ VAE 完成 ({time.time()-t0:.1f}s)")
        self._report("VAE 焊死后")

    def _load_te(self, te_name):
        self._log("\n[临时] 加载 Text Encoder...")
        t0 = time.time()
        loader = _gguf_nodes.CLIPLoaderGGUF()
        (self.clip,) = loader.load_clip(te_name, type="llm")
        clip_patcher = getattr(self.clip, 'patcher', self.clip)
        _mm.load_models_gpu([clip_patcher])
        self._log(f"   ✅ TE 加载完成 ({time.time()-t0:.1f}s)")
        self._report("TE 加载后")

    def _destroy_te(self):
        if self.clip is None:
            return
        self._log("\n[释放] 🔪 真·销毁 Text Encoder...")
        try:
            if hasattr(_mm, 'current_loaded_models'):
                clip_patcher = getattr(self.clip, 'patcher', None)
                if clip_patcher:
                    _mm.current_loaded_models = [
                        m for m in _mm.current_loaded_models
                        if m != clip_patcher
                    ]
        except Exception:
            pass
        for attr in ['cond_stage_model', 'patcher', 'model', 'transformer']:
            try:
                if hasattr(self.clip, attr):
                    obj = getattr(self.clip, attr)
                    if hasattr(obj, 'to'):
                        obj.to('cpu')
                    delattr(self.clip, attr)
            except Exception:
                pass
        del self.clip
        self.clip = None
        gc.collect(); gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        self._report("TE 释放后 (VRAM 应停留在 Trans+VAE 水位)")

    def _detach_cond(self, cond):
        if cond is None:
            return None
        def detach_val(v):
            if isinstance(v, torch.Tensor): return v.detach().clone()
            elif isinstance(v, dict):       return {kk: detach_val(vv) for kk, vv in v.items()}
            elif isinstance(v, list):       return [detach_val(vv) for vv in v]
            return v
        detached = []
        for tensor, params in cond:
            if isinstance(tensor, torch.Tensor):
                tensor = tensor.detach().clone()
            new_params = {k: detach_val(v) for k, v in params.items()}
            detached.append((tensor, new_params))
        return detached

    def encode_prompt(self, te_name, prompt, negative=""):
        self._load_te(te_name)
        from nodes import CLIPTextEncode
        encoder = CLIPTextEncode()
        (positive,)      = encoder.encode(self.clip, prompt)
        (negative_cond,) = encoder.encode(self.clip, negative)
        positive      = self._detach_cond(positive)
        negative_cond = self._detach_cond(negative_cond)
        self._destroy_te()
        return positive, negative_cond

    def generate(self, te_name, prompt, negative="",
                 width=1024, height=1024,
                 steps=4, cfg=1.0, seed=-1,
                 use_tiled_vae=False, tile_size=512):
        if seed < 0:
            seed = random.randint(0, 2**31)

        t_total = time.time()
        self._log(f"\n{'═'*60}")
        self._log(f"🎨 {width}×{height}  steps={steps}  cfg={cfg}  seed={seed}")
        if use_tiled_vae:
            self._log(f"   🧩 Tiled VAE: tile_size={tile_size}")
        self._log(f"{'═'*60}")

        t0 = time.time()
        positive, negative_cond = self.encode_prompt(te_name, prompt, negative)
        self._log(f"   ⏱️  编码: {time.time()-t0:.1f}s")

        self._log("\n[采样] DiT 采样中...")
        self._report("采样开始")
        t0 = time.time()
        from nodes import KSampler, EmptyLatentImage
        (latent,) = EmptyLatentImage().generate(width, height, batch_size=1)
        (result,) = KSampler().sample(
            model        = self.model,
            seed         = seed,
            steps        = steps,
            cfg          = cfg,
            sampler_name = "euler",
            scheduler    = "sgm_uniform",
            positive     = positive,
            negative     = negative_cond,
            latent_image = latent,
            denoise      = 1.0,
        )
        self._log(f"   ⏱️  采样: {time.time()-t0:.1f}s")
        self._report("采样完成")

        del positive, negative_cond, latent
        torch.cuda.empty_cache()

        self._log("\n[解码] VAE 解码中...")
        t0 = time.time()
        if use_tiled_vae:
            images = self.tiled_decode(result, tile_size=tile_size)
        else:
            from nodes import VAEDecode
            (images,) = VAEDecode().decode(self.vae, result)
        self._log(f"   ⏱️  解码: {time.time()-t0:.1f}s")

        del result
        torch.cuda.empty_cache()
        self._log(f"\n🎉 总耗时: {time.time()-t_total:.1f}s")
        self._report("生成完成")
        return images

    def tiled_decode(self, samples, tile_size=512):
        from nodes import VAEDecode
        try:
            from nodes import VAEDecodeTiled
            (images,) = VAEDecodeTiled().decode(self.vae, samples, tile_size)
            self._log(f"   ✅ 使用 ComfyUI 内置 VAEDecodeTiled (tile={tile_size})")
            return images
        except ImportError:
            pass

        self._log(f"   🔧 手动分块 VAE 解码 (tile={tile_size})")
        latent = samples["samples"]
        b, c, h, w = latent.shape
        lt      = tile_size // 8
        overlap = lt // 4
        stride  = lt - overlap
        tiles_h = max(1, (h - overlap + stride - 1) // stride)
        tiles_w = max(1, (w - overlap + stride - 1) // stride)
        self._log(f"   Latent {h}×{w} → {tiles_h}×{tiles_w} 块 (lt={lt}, overlap={overlap})")

        output = torch.zeros(b, h * 8, w * 8, 3, device='cpu')
        weight = torch.zeros(b, h * 8, w * 8, 1, device='cpu')

        def make_blend_mask(th, tw, overlap_px):
            mask = torch.ones(th, tw)
            if overlap_px > 0:
                for i in range(overlap_px):
                    alpha = i / overlap_px
                    mask[i, :] *= alpha;    mask[-(i+1), :] *= alpha
                    mask[:, i] *= alpha;    mask[:, -(i+1)] *= alpha
            return mask.unsqueeze(0).unsqueeze(-1)

        total = tiles_h * tiles_w
        count = 0
        for ti in range(tiles_h):
            for tj in range(tiles_w):
                count += 1
                y0 = min(ti * stride, h - lt)
                x0 = min(tj * stride, w - lt)
                y1, x1 = y0 + lt, x0 + lt
                tile_latent  = latent[:, :, y0:y1, x0:x1].clone()
                (tile_images,) = VAEDecode().decode(self.vae, {"samples": tile_latent})
                tile_img = tile_images.cpu()
                th2, tw2 = tile_img.shape[1], tile_img.shape[2]
                mask = make_blend_mask(th2, tw2, overlap * 8)
                output[:, y0*8:y0*8+th2, x0*8:x0*8+tw2, :] += tile_img * mask
                weight[:, y0*8:y0*8+th2, x0*8:x0*8+tw2, :] += mask
                del tile_latent, tile_images, tile_img
                torch.cuda.empty_cache()
                self._log(f"      块 {count}/{total} 完成")

        self._log(f"      全部 {total} 块解码完成")
        return output / torch.clamp(weight, min=1e-6)

    def vram_status(self):
        self._report("手动查询")

    def destroy(self):
        freed = []
        for attr in ['model', 'vae', 'clip']:
            if getattr(self, attr) is not None:
                try:
                    obj = getattr(self, attr)
                    nn_m = (obj.model if hasattr(obj, 'model') else
                            obj.first_stage_model if hasattr(obj, 'first_stage_model')
                            else obj)
                    if hasattr(nn_m, '__dict__') and 'to' in nn_m.__dict__:
                        del nn_m.__dict__['to']
                    nn_m.to('cpu')
                except Exception:
                    pass
                setattr(self, attr, None)
                freed.append(attr)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.synchronize()
        return freed


# ══════════════════════════════════════════════════════════════
# 🔄 ZImageEngineSerial（串行按需模式，新增）
# ══════════════════════════════════════════════════════════════
class ZImageEngineSerial(ZImageEngine):
    """
    串行按需模式：VAE 常驻焊死，Trans/TE 按需轮流上场，永不共存显存。

    生成流程（每张图）：
        ① TE 加载 → 编码 → TE 即抛
        ② Trans 流式加载（mmap + MADV_DONTNEED） → 采样 → Trans 即抛
        ③ VAE 解码（常驻，无须重载）

    适用场景：
        F16 Trans (12.3GB) + 高精 TE (4GB+) + LoRA，T4 15GB VRAM 稳跑
        ※ F16 Trans 加载时依赖 OS 对 mmap 文件页的自动驱逐；
          加载完成后立刻 MADV_DONTNEED，物理页归还速度取决于 OS 调度。

    代价：
        每张图额外 30~90s Trans 加载时间（取决于 Colab 磁盘带宽）

    Cell 2 兼容：
        _default_te / _te_name 均已设置；
        但 Cell 2 run_generation 依赖 _engine.model 常驻 ——
        串行模式的 Cell 2 适配将在第二步改造中完成。
    """
    _MODE = "serial"

    def __init__(self, output_widget=None):
        super().__init__(output_widget)
        self._unet_name  = None
        self._te_name    = None
        self._vae_path   = None

    # ── load()：只焊死 VAE，记录文件名供按需加载 ─────────────
    def load(self, unet_name, te_name, vae_path, output_widget=None):
        if output_widget:
            self._out = output_widget
        self.device = _mm.get_torch_device()

        self._unet_name       = unet_name
        self._te_name         = te_name
        self._vae_path        = vae_path
        self._default_te      = te_name   # Cell 2 兼容
        self._trans_on_demand = True      # 告知出图面板使用串行时序

        self._log(f"\n{'='*60}")
        self._log("🔄 ZImageEngineSerial 初始化（Trans/TE 串行按需版）")
        self._log(f"   设备    : {self.device}")
        self._log(f"   Trans   : {unet_name}  ← 每图按需加载+即抛")
        self._log(f"   TE      : {te_name}    ← 每图按需加载+即抛")
        self._log(f"   ⚠️  每张图额外 ~30-90s Trans 加载时间")
        self._log(f"{'='*60}")

        self._report("初始状态")
        self._load_vae(vae_path)   # 只焊死 VAE

        self._log("\n✅ VAE 已焊死。Trans/TE 将按需串行加载，互不共存。")
        self._report("初始化完成（仅 VAE 常驻）")

    # ── Trans 流式加载 ────────────────────────────────────────
    def _stream_load_transformer(self):
        """
        流式加载 Trans：
          1. 清空 mmap 注册表，确保本次追踪干净
          2. 调用 UnetLoaderGGUF（内部用 np.memmap；OS 可驱逐文件页）
          3. deep_weld 焊死显存
          4. MADV_DONTNEED 归还 GGUF mmap 物理页 + malloc_trim

        F16 GGUF 行为：HIGH_VRAM 将张量推 GPU 后，mmap 文件页即成冗余；
        MADV_DONTNEED 告知内核可立刻回收，CPU RAM 快速回落。

        量化 GGUF（Q8/Q4）行为：量化权重留在 CPU 供推理时反量化；
        此时 mmap 不能归还，_flush_gguf_mmaps 会跳过（freed_gb ≈ 0），
        内存表现与电焊模式类似——这种情况应改用电焊模式。
        """
        self._log(f"\n[串行] 流式加载 Transformer: {self._unet_name}")
        self._report("Trans 加载前（观察 CPU RAM）")
        t0 = time.time()

        # 清空上次残留；追踪器已在 _init_comfy 中安装
        _gguf_mmap_registry.clear()

        loader = _gguf_nodes.UnetLoaderGGUF()
        (self.model,) = loader.load_unet(self._unet_name)

        # 焊死显存，切断 offload 通道
        self._deep_weld("Transformer[Serial]", self.model)

        # 归还 GGUF mmap 物理页
        freed_gb = _flush_gguf_mmaps()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.time() - t0
        self._log(
            f"   ✅ Trans 流式加载完成 ({elapsed:.1f}s)"
            f"，mmap 归还 {freed_gb:.2f} GB 物理页"
        )
        self._report("Trans 加载后（CPU RAM 应回落）")

    def _destroy_transformer(self):
        """真·销毁 Trans：直接切断引用释放显存，绝不做 CPU 迁移

        注意：GGUF Q8 Trans 体积 8-10GB，obj.to('cpu') 迁移过程中
        GPU+CPU 需同时持有权重，RAM 必爆。正确做法是直接摘除引用，
        交由 GC + empty_cache 归还 CUDA 显存页。
        """
        if self.model is None:
            return
        self._log("\n[释放] 🔪 销毁 Transformer（直接切断引用，不迁移 CPU）...")

        # 1. 从 ComfyUI 调度队列摘除，防止框架持有引用
        try:
            if hasattr(_mm, 'current_loaded_models'):
                model_patcher = getattr(self.model, 'patcher', None)
                if model_patcher:
                    _mm.current_loaded_models = [
                        m for m in _mm.current_loaded_models
                        if m != model_patcher
                    ]
        except Exception:
            pass

        # 2. 逐层摘除子属性引用，不调用 .to()
        for attr in ['model', 'patcher', 'inner_model', 'object_patches']:
            try:
                if hasattr(self.model, attr):
                    delattr(self.model, attr)
            except Exception:
                pass

        # 3. 清除顶层引用
        del self.model
        self.model = None

        # 4. 强制回收：GC 清引用计数 → CUDA 归还显存页
        gc.collect()
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        try: ctypes.CDLL("libc.so.6").malloc_trim(0)
        except: pass

        self._report("Transformer 释放后 (VRAM 应仅剩 VAE 水位)")

    # ── generate()：串行编排 ──────────────────────────────────
    def generate(self, te_name, prompt, negative="",
                 width=1024, height=1024,
                 steps=4, cfg=1.0, seed=-1,
                 use_tiled_vae=False, tile_size=512):

        if seed < 0:
            seed = random.randint(0, 2**31)

        t_total = time.time()
        self._log(f"\n{'═'*60}")
        self._log(f"🎨 串行生成  {width}×{height}  steps={steps}  cfg={cfg}  seed={seed}")
        self._log(f"   流程：TE→编码→TE抛 ⟶ Trans→采样→Trans抛 ⟶ VAE解码")
        self._log(f"{'═'*60}")

        actual_te = te_name or self._te_name

        # ── Phase 1: TE 加载 → 编码 → TE 即抛 ─────────────────
        t0 = time.time()
        self._log("\n[串行 Phase 1] TE 加载 → 编码 → 即抛")
        positive, negative_cond = self.encode_prompt(actual_te, prompt, negative)
        self._log(f"   ⏱️  TE 阶段完成: {time.time()-t0:.1f}s")
        self._report("TE 抛出后（VRAM 应仅剩 VAE）")

        # ── Phase 2: Trans 流式加载 → 采样 → Trans 即抛 ────────
        self._log("\n[串行 Phase 2] Trans 流式加载 → 采样 → 即抛")
        t0 = time.time()
        self._stream_load_transformer()

        self._log("\n[串行] DiT 采样中...")
        from nodes import KSampler, EmptyLatentImage
        (latent,) = EmptyLatentImage().generate(width, height, batch_size=1)
        (result,) = KSampler().sample(
            model        = self.model,
            seed         = seed,
            steps        = steps,
            cfg          = cfg,
            sampler_name = "euler",
            scheduler    = "sgm_uniform",
            positive     = positive,
            negative     = negative_cond,
            latent_image = latent,
            denoise      = 1.0,
        )
        del positive, negative_cond, latent
        torch.cuda.empty_cache()
        self._log(f"   ⏱️  Trans+采样: {time.time()-t0:.1f}s")

        # 采样完毕，立刻销毁 Trans
        self._destroy_transformer()

        # ── Phase 3: VAE 解码（常驻，无须重载）─────────────────
        self._log("\n[串行 Phase 3] VAE 解码（常驻，无须重载）")
        t0 = time.time()
        if use_tiled_vae:
            images = self.tiled_decode(result, tile_size=tile_size)
        else:
            from nodes import VAEDecode
            (images,) = VAEDecode().decode(self.vae, result)
        del result
        torch.cuda.empty_cache()
        self._log(f"   ⏱️  VAE 解码: {time.time()-t0:.1f}s")

        self._log(f"\n🎉 串行总耗时: {time.time()-t_total:.1f}s")
        self._report("串行生成完成")
        return images

    # ── destroy()：串行模式也需要清理 VAE ──────────────────────
    def destroy(self):
        # Trans 在每图结束时已销毁；这里主要清 VAE 和残余 clip
        freed = super().destroy()
        # 确保 mmap 注册表也被清空
        _gguf_mmap_registry.clear()
        return freed


# ══════════════════════════════════════════════════════════════
# 全局引擎实例（单例）
# ══════════════════════════════════════════════════════════════
_engine: ZImageEngine = None


# ══════════════════════════════════════════════════════════════
# 🛠 工具函数
# ══════════════════════════════════════════════════════════════
def scan_local_dir(path):
    """返回列表 [(文件名, 字节数, 格式化大小字符串), ...]，过滤掉大小为0的文件"""
    result = []
    if not os.path.isdir(path):
        return result
    for f in sorted(os.listdir(path)):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            size = os.path.getsize(fp)
            if size == 0:
                continue  # 不显示0字节占位文件
            size_str = (f"{size/1024**3:.2f} GB" if size > 1024**2
                        else f"{size/1024:.0f} KB")
            result.append((f, size, size_str))
    return result

def fetch_repo_files(repo_id, exts=('.gguf', '.safetensors')):
    try:
        from huggingface_hub import list_repo_files
        files = list_repo_files(repo_id)
        return sorted([f for f in files if any(f.endswith(e) for e in exts)])
    except Exception as e:
        return [f"(ERROR: {e})"]

def download_file_flat(repo_id, filename, local_dir, output_widget):
    from huggingface_hub import hf_hub_download
    target_name = filename.split('/')[-1]
    final_path  = os.path.join(local_dir, target_name)
    if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
        with output_widget:
            print(f"   ⏭  已存在，跳过: {target_name}")
        return True
    try:
        with output_widget:
            print(f"   📥 下载中: {filename} ...")
        downloaded = hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=local_dir, local_dir_use_symlinks=False
        )
        if downloaded != final_path and os.path.exists(downloaded):
            os.rename(downloaded, final_path)
        with output_widget:
            print(f"   ✅ 完成: {target_name}")
        return True
    except Exception as e:
        with output_widget:
            print(f"   ❌ 失败: {e}")
        return False

def format_vram():
    if not torch.cuda.is_available():
        return "CPU 模式"
    alloc = torch.cuda.memory_allocated() / 1024**3
    resv  = torch.cuda.memory_reserved()  / 1024**3
    return f"显存: {alloc:.2f}GB 已用 / {resv:.2f}GB 预留"


# ══════════════════════════════════════════════════════════════
# 🎨 UI 构建
# ══════════════════════════════════════════════════════════════
S_BTN = widgets.Layout(width="auto", height="32px")

header = widgets.HTML("""
<div style="padding:8px 0 10px; border-bottom:1px solid #334155; margin-bottom:10px;">
  <b style="font-size:14px; color:#38bdf8; font-family:monospace;">
    🤖 ComfyUI Model Manager — 双模式版（电焊 + 串行按需）
  </b><br>
  <span style="font-size:11px; color:#64748b; font-family:monospace;">
    Trans/VAE 常驻显存 · TE 真·即抛 · Tiled VAE · 免疫 Offload
    &nbsp;|&nbsp; 串行模式：Trans/TE 轮流上场，永不共存
  </span>
</div>
""")

scan_html = widgets.HTML(value="<i style='color:gray;font-size:12px;'>正在扫描...</i>")

def build_scan_html():
    rows = []
    for label, path in [("unet", DIR_UNET), ("clip", DIR_CLIP), ("vae", DIR_VAE)]:
        files = scan_local_dir(path)   # 已经过滤掉0字节文件
        if files:
            items = "".join(
                f"<tr><td style='padding:1px 6px 1px 0;font-family:monospace;"
                f"font-size:11px;color:#22c55e;'>● {f}</td>"
                f"<td style='font-size:10px;color:#64748b;font-family:monospace;'>{s}</td></tr>"
                for f, _, s in files
            )
        else:
            items = ("<tr><td colspan=2 style='font-size:11px;color:#64748b;"
                     "font-family:monospace;padding-left:4px;'>— 空目录</td></tr>")
        rows.append(
            f"<td style='vertical-align:top;padding-right:16px;'>"
            f"<div style='font-size:10px;color:#94a3b8;font-family:monospace;"
            f"margin-bottom:2px;'>{label}/</div><table>{items}</td></td>"
        )
    return f"<table><tr>{''.join(rows)}</tr></table>"

rescan_btn = widgets.Button(
    description="🔄 刷新本地文件", layout=S_BTN,
    style={'button_color': '#1e293b', 'font_weight': 'bold'}
)
def do_rescan(_=None):
    scan_html.value = build_scan_html()
    try:
        # 获取过滤掉0字节的文件名列表
        local_unets = [f for f, sz, _ in scan_local_dir(DIR_UNET) if sz > 0]
        unet_file_select.options = ([(f"✅ {f} [本地]", f) for f in local_unets]
                                    if local_unets else [("(无本地文件，请Browse)", "")])
        local_tes = [f for f, sz, _ in scan_local_dir(DIR_CLIP) if sz > 0]
        te_file_select.options = ([(f"✅ {f} [本地]", f) for f in local_tes]
                                   if local_tes else [("(无本地文件，请Browse)", "")])
        local_vaes = [f for f, sz, _ in scan_local_dir(DIR_VAE) if sz > 0]
        vae_file_select.options = (
            [("— 自动: ae.safetensors —", "__auto__")]
            + [(f"✅ {f} [本地]", f) for f in local_vaes]
        )
    except NameError:
        pass

rescan_btn.on_click(do_rescan)

# ── Unet 区 ──
unet_repo_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("unsloth/Z-Image-Turbo-GGUF", "unsloth/Z-Image-Turbo-GGUF"),
        ("unsloth/Z-Image-GGUF", "unsloth/Z-Image-GGUF"),
        ("unsloth/ERNIE-Image-Turbo-GGUF", "unsloth/ERNIE-Image-Turbo-GGUF"),
    ],
    value=DEFAULT_REPO_UNET,
    layout=widgets.Layout(width="200px")
)
unet_repo_input = widgets.Text(
    value=DEFAULT_REPO_UNET, placeholder="HuggingFace repo id",
    description="Repo:", style={'description_width': '40px'},
    layout=widgets.Layout(flex="1 1 auto")
)

def _on_unet_preset_change(change):
    if change['new']:
        unet_repo_input.value = change['new']
unet_repo_preset.observe(_on_unet_preset_change, names='value')

unet_browse_btn = widgets.Button(description="Browse", layout=widgets.Layout(width="80px", height="32px"))
unet_repo_row = widgets.HBox([unet_repo_preset, unet_repo_input, unet_browse_btn], layout=widgets.Layout(width="100%"))
unet_file_select = widgets.Dropdown(
    options=["(点击 Browse 加载列表)"],
    description="文件:", style={'description_width': '30px'},
    layout=widgets.Layout(width="100%")
)
def unet_browse(_):
    unet_file_select.options = ["⏳ 加载中..."]
    files = fetch_repo_files(unet_repo_input.value)
    # 获取本地非0字节文件集合
    local = {f for f, sz, _ in scan_local_dir(DIR_UNET) if sz > 0}
    unet_file_select.options = [
        (f"✅ {f.split('/')[-1]} [本地]" if f.split('/')[-1] in local
         else f.split('/')[-1], f)
        for f in files
    ]
unet_browse_btn.on_click(unet_browse)

# ── Text Encoder 区 ──
te_repo_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("unsloth/Qwen3-4B-GGUF", "unsloth/Qwen3-4B-GGUF"),
        ("unsloth/Ministral-3-3B-Instruct-2512-GGUF", "unsloth/Ministral-3-3B-Instruct-2512-GGUF"),
    ],
    value=DEFAULT_REPO_TE,
    layout=widgets.Layout(width="200px")
)
te_repo_input = widgets.Text(
    value=DEFAULT_REPO_TE, placeholder="HuggingFace repo id",
    description="Repo:", style={'description_width': '40px'},
    layout=widgets.Layout(flex="1 1 auto")
)

def _on_te_preset_change(change):
    if change['new']:
        te_repo_input.value = change['new']
te_repo_preset.observe(_on_te_preset_change, names='value')

te_browse_btn = widgets.Button(description="Browse", layout=widgets.Layout(width="80px", height="32px"))
te_repo_row = widgets.HBox([te_repo_preset, te_repo_input, te_browse_btn], layout=widgets.Layout(width="100%"))
te_file_select  = widgets.Dropdown(
    options=["(点击 Browse 加载列表)"],
    description="文件:", style={'description_width': '30px'},
    layout=widgets.Layout(width="100%")
)
def te_browse(_):
    te_file_select.options = ["⏳ 加载中..."]
    files = fetch_repo_files(te_repo_input.value)
    local = {f for f, sz, _ in scan_local_dir(DIR_CLIP) if sz > 0}
    te_file_select.options = [
        (f"✅ {f.split('/')[-1]} [本地]" if f.split('/')[-1] in local
         else f.split('/')[-1], f)
        for f in files
    ]
te_browse_btn.on_click(te_browse)

# ── VAE 槽位 ──
vae_repo_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("Comfy-Org/z_image_turbo", "Comfy-Org/z_image_turbo"),
        ("black-forest-labs/FLUX.1-dev", "black-forest-labs/FLUX.1-dev"),
        ("Comfy-Org/ERNIE-Image", "Comfy-Org/ERNIE-Image"),
    ],
    value="",
    layout=widgets.Layout(width="200px")
)
vae_repo_input = widgets.Text(
    value="", placeholder=f"留空 = {DEFAULT_REPO_VAE}",
    description="Repo:", style={'description_width': '40px'},
    layout=widgets.Layout(flex="1 1 auto")
)

def _on_vae_preset_change(change):
    if change['new']:
        vae_repo_input.value = change['new']
    else:
        vae_repo_input.value = ""
vae_repo_preset.observe(_on_vae_preset_change, names='value')

vae_browse_btn = widgets.Button(description="Browse", layout=widgets.Layout(width="80px", height="32px"))
vae_repo_row = widgets.HBox([vae_repo_preset, vae_repo_input, vae_browse_btn], layout=widgets.Layout(width="100%"))
vae_file_select = widgets.Dropdown(
    options=["— 自动: ae.safetensors —"],
    description="文件:", style={'description_width': '30px'},
    layout=widgets.Layout(width="100%")
)
def vae_browse(_):
    repo = vae_repo_input.value.strip() or DEFAULT_REPO_VAE
    vae_file_select.options = ["⏳ 加载中..."]
    files = fetch_repo_files(repo, exts=('.safetensors', '.gguf', '.bin'))
    local = {f for f, sz, _ in scan_local_dir(DIR_VAE) if sz > 0}
    vae_file_select.options = [("— 自动: ae.safetensors —", "__auto__")] + [
        (f"✅ {f.split('/')[-1]} [本地]" if f.split('/')[-1] in local
         else f.split('/')[-1], f)
        for f in files
    ]
vae_browse_btn.on_click(vae_browse)

# ── 操作模式（新增 e: 串行按需）────────────────────────────────
mode_radio = widgets.RadioButtons(
    options=[
        ("a: 下载文件    (检查缺失并从 HuggingFace 下载)",                      "download"),
        ("b: 电焊加载    (Trans/VAE 锁死显存，TE 即抛 · 量化模型首选)",          "load"),
        ("c: 检查本地    (仅扫描并打印文件状态)",                                "inspect"),
        ("d: 清除引擎    (恢复 .to() 后释放全部显存)",                           "purge"),
        ("e: 串行按需    (Trans/TE 轮流上场永不共存 · F16+高精TE)", "serial"),
    ],
    value="download",
    layout=widgets.Layout(width="100%")
)

# ── 串行模式说明提示（动态显隐）────────────────────────────────
serial_hint = widgets.HTML(
    value="""<div style="margin:6px 0 0 20px; padding:10px 14px;
                border-left:3px solid #f59e0b; background:#1c1a10;
                border-radius:0 8px 8px 0; font-family:monospace; font-size:12px;">
      <b style="color:#f59e0b;">⚡ 串行按需模式说明</b><br>
      <span style="color:#d4b96a;">
      · 只焊死 VAE（~1GB），Trans/TE 每图按需轮流上场，永不同时在显存<br>
      · F16 Trans (12.3GB) + 高精 TE (4GB+) + LoRA → 总峰值 ≈ 14GB，T4 可跑<br>
      · 加载机制：GGUF 内部用 np.memmap；加载完立刻 MADV_DONTNEED 归还物理页<br>
      · 代价：每张图额外 <b style="color:#fb923c;">30~90s</b> 的 Trans 磁盘加载时间<br>
      · ⚠️ Cell 2 出图面板需第二步改造方可使用串行模式；此步完成后可通过底部
        <code>generate()</code> 函数验证
      </span>
    </div>""",
    layout=widgets.Layout(display="none")
)

def _on_mode_change(c):
    serial_hint.layout.display = "block" if c["new"] == "serial" else "none"
mode_radio.observe(_on_mode_change, names="value")

# ── 输出与状态 ──
output   = widgets.Output()
status_w = widgets.HTML("<span style='font-size:12px;color:gray;font-family:monospace;'>就绪</span>")

exec_btn  = widgets.Button(
    description="🚀 执行", button_style="primary",
    layout=widgets.Layout(width="100px", height="36px")
)
purge_btn = widgets.Button(
    description="💀 清除引擎", button_style="danger",
    layout=widgets.Layout(width="130px", height="36px")
)
do_rescan()


# ══════════════════════════════════════════════════════════════
# ⚙️  执行逻辑
# ══════════════════════════════════════════════════════════════
def get_selected_file(dropdown):
    v = dropdown.value
    return v if v else None

def on_exec(_):
    global _engine
    exec_btn.disabled = purge_btn.disabled = True
    output.clear_output()
    mode = mode_radio.value

    unet_repo = unet_repo_input.value.strip()
    te_repo   = te_repo_input.value.strip()
    vae_repo  = vae_repo_input.value.strip() or DEFAULT_REPO_VAE

    unet_file = get_selected_file(unet_file_select)
    te_file   = get_selected_file(te_file_select)
    vae_file  = get_selected_file(vae_file_select)
    if not vae_file or vae_file in ("__auto__",) or vae_file.startswith("—"):
        vae_file = DEFAULT_VAE_PATH
        vae_repo = DEFAULT_REPO_VAE

    try:
        # ── c: 检查本地 ──────────────────────────────────────
        if mode == "inspect":
            status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>🔍 扫描中...</span>"
            with output:
                print("=" * 55)
                print("📁 本地文件检查报告")
                print("=" * 55)
                for label, path in [("unet", DIR_UNET), ("clip", DIR_CLIP), ("vae", DIR_VAE)]:
                    files = scan_local_dir(path)
                    print(f"\n📂 {path}")
                    if files:
                        for f, sz, s in files:
                            print(f"   ✅ {f}  ({s})")
                    else:
                        print("   — 无有效文件（0KB占位文件已过滤）")
                print("\n" + format_vram())
            do_rescan()
            status_w.value = "<span style='color:green;font-family:monospace;font-size:12px;'>✅ 检查完成</span>"

        # ── a: 下载 ───────────────────────────────────────────
        elif mode == "download":
            status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>⏳ 下载中...</span>"
            with output:
                print("=" * 55)
                print("📦 开始下载核心组件")
                print("=" * 55)
                if not unet_file or unet_file.startswith("("):
                    print("⚠️  未选择 Unet 文件，跳过")
                else:
                    fname = unet_file.split('/')[-1]
                    print(f"\n🔹 Unet: {fname}  ({unet_repo})")
                    download_file_flat(unet_repo, unet_file, DIR_UNET, output)
                if not te_file or te_file.startswith("("):
                    print("⚠️  未选择 TE 文件，跳过")
                else:
                    fname = te_file.split('/')[-1]
                    print(f"\n🔹 TE: {fname}  ({te_repo})")
                    download_file_flat(te_repo, te_file, DIR_CLIP, output)
                print(f"\n🔹 VAE: {vae_file.split('/')[-1]}  ({vae_repo})")
                download_file_flat(vae_repo, vae_file, DIR_VAE, output)
                print(f"\n{format_vram()}")
            do_rescan()
            status_w.value = "<span style='color:green;font-family:monospace;font-size:12px;'>✅ 下载完成</span>"

        # ── b: 电焊加载 ───────────────────────────────────────
        elif mode == "load":
            status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>⏳ 电焊加载中...</span>"
            with output:
                print("=" * 55)
                print("⚡ 电焊加载：物理锁定 Trans/VAE，TE 即抛模式")
                print("=" * 55)

                if not _init_comfy(output_widget=output):
                    print("❌ ComfyUI 运行时初始化失败，终止")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ 初始化失败</span>"
                    return

                unet_name = unet_file.split('/')[-1] if unet_file and not unet_file.startswith("(") else None
                te_name   = te_file.split('/')[-1]   if te_file   and not te_file.startswith("(")   else None
                vae_name  = vae_file.split('/')[-1]
                vae_full_path = os.path.join(DIR_VAE, vae_name)

                if not os.path.exists(vae_full_path):
                    print(f"❌ VAE 文件不存在: {vae_full_path}")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ VAE 不存在</span>"
                    return
                if not unet_name:
                    print("⚠️  未选择 Unet 文件，终止")
                    return

                if _engine is not None:
                    print("♻️  检测到旧引擎，先销毁...")
                    _engine.destroy()

                _engine = ZImageEngine(output_widget=output)
                _engine.load(
                    unet_name=unet_name, te_name=te_name,
                    vae_path=vae_full_path, output_widget=output
                )
                _engine._default_te = te_name

            status_w.value = "<span style='color:#22c55e;font-family:monospace;font-size:12px;'>✅ 电焊完成 — 可开始生成</span>"

        # ── e: 串行按需加载 ───────────────────────────────────  ← 新增
        elif mode == "serial":
            status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>⏳ 串行按需初始化中...</span>"
            with output:
                print("=" * 55)
                print("🔄 串行按需：VAE 焊死，Trans/TE 每图轮流上场")
                print("=" * 55)

                if not _init_comfy(output_widget=output):
                    print("❌ ComfyUI 运行时初始化失败，终止")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ 初始化失败</span>"
                    return

                unet_name = unet_file.split('/')[-1] if unet_file and not unet_file.startswith("(") else None
                te_name   = te_file.split('/')[-1]   if te_file   and not te_file.startswith("(")   else None
                vae_name  = vae_file.split('/')[-1]
                vae_full_path = os.path.join(DIR_VAE, vae_name)

                # 文件存在性检查
                if not os.path.exists(vae_full_path):
                    print(f"❌ VAE 文件不存在: {vae_full_path}")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ VAE 不存在</span>"
                    return
                if not unet_name:
                    print("⚠️  未选择 Trans 文件，终止")
                    status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>⚠️ 未选 Trans</span>"
                    return
                if not te_name:
                    print("⚠️  未选择 TE 文件，终止")
                    status_w.value = "<span style='color:orange;font-family:monospace;font-size:12px;'>⚠️ 未选 TE</span>"
                    return

                unet_full = os.path.join(DIR_UNET, unet_name)
                te_full   = os.path.join(DIR_CLIP, te_name)
                if not os.path.exists(unet_full):
                    print(f"❌ Trans 文件不存在: {unet_full}")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ Trans 不存在</span>"
                    return
                if not os.path.exists(te_full):
                    print(f"❌ TE 文件不存在: {te_full}")
                    status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ TE 不存在</span>"
                    return

                if _engine is not None:
                    print("♻️  检测到旧引擎，先销毁...")
                    _engine.destroy()

                _engine = ZImageEngineSerial(output_widget=output)
                _engine.load(
                    unet_name=unet_name, te_name=te_name,
                    vae_path=vae_full_path, output_widget=output
                )

                print(f"\n📌 引擎类型 : {_engine._MODE}")
                print(f"   Trans    : {unet_name}  （每图按需加载+即抛）")
                print(f"   TE       : {te_name}  （每图按需加载+即抛）")
                print(f"   VAE      : {os.path.basename(vae_full_path)}  （常驻）")
                print(f"\n✅ 预加载完成")


            status_w.value = "<span style='color:#22c55e;font-family:monospace;font-size:12px;'>✅ 串行就绪 — 每图按需加载 Trans/TE</span>"

        # ── d: 清除引擎 ───────────────────────────────────────
        elif mode == "purge":
            output.clear_output()
            with output:
                print("=" * 55)
                print("💀 销毁引擎，释放全部显存")
                print("=" * 55)
                if _engine is not None:
                    freed = _engine.destroy()
                    _engine = None
                    print(f"✅ 已清除: {', '.join(freed) if freed else '（无已加载模型）'}")
                else:
                    print("（引擎未初始化）")
                gc.collect()
                torch.cuda.empty_cache()
                print(format_vram())
            status_w.value = "<span style='color:green;font-family:monospace;font-size:12px;'>🧹 引擎已销毁</span>"

    except Exception as e:
        import traceback
        status_w.value = "<span style='color:red;font-family:monospace;font-size:12px;'>❌ 出错</span>"
        with output:
            print(f"\n❌ 错误: {e}")
            traceback.print_exc()
    finally:
        exec_btn.disabled = purge_btn.disabled = False


def on_purge(_):
    global _engine
    output.clear_output()
    exec_btn.disabled = purge_btn.disabled = True
    try:
        with output:
            if _engine is not None:
                freed = _engine.destroy()
                _engine = None
                print(f"✅ 已清除: {', '.join(freed)}")
            else:
                print("（引擎未初始化）")
            gc.collect()
            torch.cuda.empty_cache()
            print(format_vram())
        status_w.value = "<span style='color:green;font-family:monospace;font-size:12px;'>🧹 显存已清理</span>"
    finally:
        exec_btn.disabled = purge_btn.disabled = False

exec_btn.on_click(on_exec)
purge_btn.on_click(on_purge)


# ══════════════════════════════════════════════════════════════
# 🖥️  渲染 UI
# ══════════════════════════════════════════════════════════════
def sep(label=""):
    return widgets.HTML(
        f"<div style='border-top:1px solid #1e293b;margin:8px 0 4px;"
        f"font-size:10px;color:#475569;font-family:monospace;padding-top:4px;'>{label}</div>"
    )

layout = widgets.VBox([
    header,

    widgets.HBox(
        [widgets.HTML("<span style='font-size:11px;color:#94a3b8;font-family:monospace;'>📂 本地文件</span>"),
         rescan_btn],
        layout=widgets.Layout(justify_content="space-between", align_items="center")
    ),
    scan_html,

    sep("── Unet GGUF ──"),
    unet_repo_row,
    unet_file_select,

    sep("── Text Encoder GGUF ──"),
    te_repo_row,
    te_file_select,

    sep("── VAE (留空 = 默认官方地址) ──"),
    vae_repo_row,
    vae_file_select,

    sep("── 操作模式 ──"),
    mode_radio,
    serial_hint,        # ← 新增：串行模式说明（仅选 e 时显示）

    widgets.HBox(
        [exec_btn, purge_btn, status_w],
        layout=widgets.Layout(gap="10px", align_items="center", margin="6px 0 0")
    ),
    output,

], layout=widgets.Layout(
    padding="14px", width="760px",
    border="1px solid #1e293b", border_radius="10px"
))

display(layout)


# ══════════════════════════════════════════════════════════════
# 📌 生成接口（供外部 cell 直接调用，兼容两种引擎）
# ══════════════════════════════════════════════════════════════
def generate(prompt, negative="",
             width=2048, height=2048,
             steps=8, cfg=1.0, seed=-1,
             use_tiled_vae=True, tile_size=512):
    """
    外部调用示例（两种引擎均可）：
        imgs = generate("your prompt here", width=2048, height=2048)

    电焊模式：Trans 常驻，每图仅加载/抛出 TE
    串行模式：每图依次加载/抛出 TE → 加载/抛出 Trans → VAE 解码
    """
    if _engine is None:
        raise RuntimeError("引擎未初始化，请先执行「电焊加载」或「串行按需」")

    # 兼容两种引擎的 TE 名获取
    te_name = (getattr(_engine, '_default_te', None)
               or getattr(_engine, '_te_name', None))
    if te_name is None:
        raise RuntimeError("未记录 TE 文件名，请重新初始化引擎")

    return _engine.generate(
        te_name       = te_name,
        prompt        = prompt,
        negative      = negative,
        width         = width,
        height        = height,
        steps         = steps,
        cfg           = cfg,
        seed          = seed,
        use_tiled_vae = use_tiled_vae,
        tile_size     = tile_size,
    )

In [ ]:
# -*- coding: utf-8 -*-
# @title 🍳 GGUF-LoRA 离线融合器
# ============================================================
# 优化点：
# 1. 动态槽位：初始 1 个，点击按钮可无限添加。
# 2. 纯净日志：拦截所有 "lora key not loaded" 等扰民警告。
# 3. 内存零损耗：完美继承 F16 的 r+ mmap 物理覆写魔法。
# ============================================================

import os, sys, shutil, gc, time, logging, contextlib
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

if "/content/ComfyUI" not in sys.path:
    sys.path.insert(0, '/content/ComfyUI')

import comfy.utils
import comfy.sd
import comfy.model_management as mm

# ── 拦截扰民警告的消音器 ──────────────────────────────
@contextlib.contextmanager
def suppress_lora_warnings():
    logger = logging.getLogger()
    class LoraWarningFilter(logging.Filter):
        def filter(self, record):
            msg = record.getMessage()
            # 过滤掉 ComfyUI 未命中 key 的提示
            return "lora key not loaded" not in msg

    f = LoraWarningFilter()
    logger.addFilter(f)
    try:
        yield
    finally:
        logger.removeFilter(f)

# ── 魔法核心：拦截 GGUF 强制开启写权限 ────────────────
import gguf
if not hasattr(gguf.GGUFReader, "_original_init_for_bake"):
    gguf.GGUFReader._original_init_for_bake = gguf.GGUFReader.__init__

def _rplus_init(self, path, mode='r'):
    gguf.GGUFReader._original_init_for_bake(self, path, mode='r+')

DIR_UNET = "/content/ComfyUI/models/unet"
DIR_LORA = "/content/ComfyUI/models/loras"
os.makedirs(DIR_UNET, exist_ok=True)
os.makedirs(DIR_LORA, exist_ok=True)

def get_lora_files():
    if not os.path.exists(DIR_LORA): return [("（不加载）", "")]
    return [("（不加载）", "")] +[(f, f) for f in sorted(os.listdir(DIR_LORA)) if f.endswith(('.safetensors', '.pt'))]

def get_unet_files():
    if not os.path.exists(DIR_UNET): return []
    return sorted([f for f in os.listdir(DIR_UNET) if f.endswith('.gguf')])

# ── UI 构建与动态逻辑 ──────────────────────────────────
out_bake = widgets.Output()

header_html = widgets.HTML("""
<div style="background:#1e1b4b; padding:15px; border-radius:10px; border:1px solid #4338ca; margin-bottom:10px;">
    <h3 style="color:#a5b4fc; margin:0 0 5px 0;">🍳 GGUF-LoRA 离线融合器 </h3>
    <span style="color:#818cf8; font-size:12px;">支持无限追加 LoRA，无报错刷屏，突破 F16 OOM 极限的终极法宝。</span>
</div>
""")

dd_unet = widgets.Dropdown(options=get_unet_files(), description='基础GGUF:', layout=widgets.Layout(width='100%'))
text_out = widgets.Text(description='输出命名:', placeholder='例如: baked_mix.gguf', layout=widgets.Layout(width='100%'))

lora_slots = []
slots_container = widgets.VBox([])

def _update_out_name(*args):
    if not dd_unet.value: return
    base = dd_unet.value.replace('.gguf', '')
    active_count = sum(1 for dd, _ in lora_slots if dd.value and dd.value != "")
    if active_count == 1:
        first_lora = [dd.value.split('.')[0] for dd, _ in lora_slots if dd.value][0]
        text_out.value = f"{base}_baked_{first_lora}.gguf"
    elif active_count > 1:
        text_out.value = f"{base}_baked_mix_{active_count}.gguf"

dd_unet.observe(_update_out_name, 'value')

def add_slot(b=None):
    idx = len(lora_slots) + 1
    opts = get_lora_files()
    dd = widgets.Dropdown(options=opts, value="", description=f"LoRA槽{idx}:", layout=widgets.Layout(width='60%'))
    sm = widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.05, description='强度:', layout=widgets.Layout(width='38%'))

    dd.observe(_update_out_name, 'value')

    row = widgets.HBox([dd, sm])
    lora_slots.append((dd, sm))
    slots_container.children = list(slots_container.children) + [row]
    _update_out_name()

# 初始给 1 个槽位
add_slot()

btn_add_slot = widgets.Button(description='➕ 添加 LoRA', button_style='info', layout=widgets.Layout(width='120px', height='32px'))
btn_add_slot.on_click(add_slot)

btn_bake = widgets.Button(description='🔥 串联开始烘焙', button_style='danger', layout=widgets.Layout(width='200px', height='40px'))

# ── 核心融合引擎 ────────────────────────────────────────
def execute_bake(_):
    out_bake.clear_output()
    btn_bake.disabled = True
    btn_add_slot.disabled = True
    try:
        with out_bake:
            base_unet = dd_unet.value
            out_name  = text_out.value.strip()

            active_loras =[]
            for dd, sm in lora_slots:
                if dd.value and abs(sm.value) > 0.001:
                    active_loras.append((dd.value, sm.value))

            if not base_unet:
                print("❌ 错误：请选择基础 GGUF 模型。")
                return
            if not active_loras:
                print("❌ 错误：请至少在槽位中选择一个有效强度不为0的 LoRA。")
                return
            if not out_name:
                out_name = base_unet.replace('.gguf', '') + "_baked_multi.gguf"
            if not out_name.endswith('.gguf'): out_name += '.gguf'

            base_path = os.path.join(DIR_UNET, base_unet)
            out_path  = os.path.join(DIR_UNET, out_name)

            print("="*60)
            print(f"🚀 开始多重串烤烘焙任务 (静默模式)")
            print(f"   基础模型: {base_unet}")
            for idx, (lname, lstr) in enumerate(active_loras):
                print(f"   注入 [{idx+1}]: {lname} (强度 {lstr})")
            print(f"   输出文件: {out_name}")
            print("="*60)

            print(f"\n⏳ 1/4 拷贝底模至输出路径 (需10-30秒)...")
            t0 = time.time()
            shutil.copy2(base_path, out_path)
            print(f"   ✅ 拷贝完成！耗时 {time.time()-t0:.1f}s")

            print(f"⏳ 2/4 挂载 r+ 读写权限并拉起 Patch 系统...")
            gguf.GGUFReader.__init__ = _rplus_init
            from ComfyUI_GGUF import nodes as gguf_nodes
            loader = gguf_nodes.UnetLoaderGGUF()
            (model_patcher,) = loader.load_unet(out_name)

            print(f"⏳ 3/4 链式解析 {len(active_loras)} 个 LoRA/LoKr 权重结构...")
            # 💡 在这里启动消音器，拦截所有烦人的 Warning
            with suppress_lora_warnings():
                for lname, lstr in active_loras:
                    lpath = os.path.join(DIR_LORA, lname)
                    print(f"   🔗 正在提取并叠加账本: {lname} ...")
                    lora_sd = comfy.utils.load_torch_file(lpath)
                    model_patcher, _ = comfy.sd.load_lora_for_models(model_patcher, None, lora_sd, lstr, 0.0)
                    del lora_sd
                    gc.collect()

            print(f"⏳ 4/4 正在进行多合一物理磁盘覆写 (Zero-RAM 模式)...")
            patches = model_patcher.patches
            total_layers = len(patches)
            count_success = 0
            count_skipped = 0

            for i, (key, layer_patches) in enumerate(patches.items()):
                if not layer_patches: continue

                raw_param = model_patcher.model.state_dict().get(key)
                if raw_param is None: continue
                raw_tensor = raw_param.data if hasattr(raw_param, 'data') else raw_param

                if raw_tensor.dtype not in[torch.float16, torch.float32]:
                    count_skipped += 1
                    continue

                try:
                    new_weight = model_patcher.calculate_weight(layer_patches, raw_tensor, key)
                    raw_tensor.copy_(new_weight.to(raw_tensor.dtype).cpu())

                    del new_weight
                    gc.collect()

                    count_success += 1
                    if count_success % 20 == 0:
                        print(f"   🔨 已融合 {count_success}/{total_layers} 个底层张量...")

                except Exception as e:
                    print(f"   ⚠️ 张量 {key} 融合失败: {e}")

            print("\n🎉 极速烘焙大功告成！")
            print(f"   ✅ 成功混合修改 {count_success} 个底层张量，保护跳过 {count_skipped} 个非F16层。")
            print("="*60)

    except Exception as e:
        print(f"\n❌ 致命异常: {e}")
        import traceback; traceback.print_exc()
    finally:
        gguf.GGUFReader.__init__ = gguf.GGUFReader._original_init_for_bake
        if 'model_patcher' in locals(): del model_patcher
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        btn_bake.disabled = False
        btn_add_slot.disabled = False

btn_bake.on_click(execute_bake)

ui = widgets.VBox([
    header_html,
    dd_unet,
    widgets.HTML("<div style='margin-top:8px;font-size:12px;color:gray;'>请配置要融合的 LoRA 列表：</div>"),
    slots_container,
    widgets.HBox([btn_add_slot], layout=widgets.Layout(margin='5px 0 15px 0')),
    widgets.HTML("<hr>"),
    text_out,
    btn_bake,
    out_bake
], layout=widgets.Layout(padding="20px", border="1px solid #475569", border_radius="10px", background="#0f172a"))

display(ui)

In [ ]:
# @title 📩 Ministral补丁（使用ERNIE-Image GGUF-TE需打补丁）
import os, urllib.request, json
import comfy.text_encoders.flux
import gguf

print("🚀 部署 ERNIE-Image & Ministral-3B 终极全内存护航 (精准匹配版)...")

# ==========================================
# 1. 内存级拦截 GGUFReader，解决 mistral3 架构识别 (免文件修改)
# ==========================================
if not hasattr(gguf.GGUFReader, "_original_init"):
    gguf.GGUFReader._original_init = gguf.GGUFReader.__init__

def patched_gguf_init(self, *args, **kwargs):
    self._original_init(*args, **kwargs)
    arch_field = self.fields.get("general.architecture")
    if arch_field and len(arch_field.data) > 0:
        try:
            idx = arch_field.data[0]
            val = arch_field.parts[idx]
            arch_str = str(val, encoding="utf-8") if isinstance(val, bytes) else str(bytes(val), encoding="utf-8")
            if arch_str in ['mistral3', 'mistral']:
                print(f"  🔧 [GGUFReader] 发现架构 '{arch_str}'，已在内存中篡改为 'llama'")
                arch_field.parts[idx] = bytearray(b"llama")
        except Exception as e:
            pass

gguf.GGUFReader.__init__ = patched_gguf_init
print("  ✅ 架构内存欺骗已就绪")

# ==========================================
# 2. 拉取真正配套的 Ministral-3B-Instruct 词表
# ==========================================
# 换个新名字，防止读到之前下错的旧文件
tekken_path = "/content/tekken_ministral_3b.json"
if not os.path.exists(tekken_path):
    print("  ⏳ 正在拉取 Ministral-3-3B-Instruct-2512 原装词表...")
    # 采用你提供的极其正确的原装模型仓库地址！
    url = "https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512/resolve/main/tekken.json"
    try:
        urllib.request.urlretrieve(url, tekken_path)
        print("  ✅ 原装词表下载完成！")
    except Exception as e:
        print(f"  ❌ 词表下载失败: {e}")

with open(tekken_path, "r", encoding="utf-8") as f:
    vocab_json = json.loads(f.read())

# 填平 ComfyUI 的特殊 Token 漏洞
if "special_tokens" not in vocab_json:
    vocab_json["special_tokens"] =[]

patched_data = json.dumps(vocab_json)
print("  ✅ 词表数据修复完毕")

# ==========================================
# 3. 内存 Hook 注入 ComfyUI
# ==========================================
if not hasattr(comfy.text_encoders.flux, "original_load_mistral"):
    comfy.text_encoders.flux.original_load_mistral = comfy.text_encoders.flux.load_mistral_tokenizer

def patched_load_mistral(data_in, *args, **kwargs):
    print("  🔧 [原装词表外挂生效] 正在注入 Ministral-3-3B-Instruct-2512 专属 Tekken 词表...")
    return comfy.text_encoders.flux.original_load_mistral(patched_data, *args, **kwargs)

comfy.text_encoders.flux.load_mistral_tokenizer = patched_load_mistral
print("🎉 Ministral补丁修补完毕")

In [ ]:
# -*- coding: utf-8 -*-
# @title 🖌️ 出图面板
import os, gc, time, random, json, numpy as np, torch, sys, ctypes, base64, io, math
import torch.nn.functional as F
from PIL import Image, PngImagePlugin, ImageFilter, ImageDraw, ImageChops
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import contextlib
import types
import inspect
import __main__

try: from google.colab import output
except ImportError: pass

# ==========================================
# 0. 检查引擎环境
# ==========================================
if "/content/ComfyUI" not in sys.path:
    sys.path.insert(0, '/content/ComfyUI')

try:
    from nodes import NODE_CLASS_MAPPINGS
    import comfy.model_management as mm
except ImportError:
    print("⚠️ 无法导入 ComfyUI 节点，请确保已安装在 /content/ComfyUI")

# ==========================================
# 1. 显存工具 & 极限卸载工具
# ==========================================
def _get_allocator_conf(): return os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "")
def _allocator_status_text():
    conf = _get_allocator_conf()
    if not conf: return "<span style='color:orange;'>⚠️ allocator: 未设置</span>"
    return f"<span style='color:green;'>✅ allocator: {conf}</span>" if "expandable_segments:True" in conf else f"<span style='color:orange;'>⚠️ allocator: {conf}</span>"
def _soft_vram_cleanup():
    if torch.cuda.is_available():
        try: torch.cuda.synchronize()
        except: pass
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
def _round_to_16(x: int) -> int: return int(round(x / 16) * 16)
def _ceil_to_16(x: float) -> int: return int(math.ceil(x / 16.0) * 16)
def _format_vram():
    if not torch.cuda.is_available(): return "CPU 模式"
    return f"显存: {torch.cuda.memory_allocated() / 1024**3:.2f}GB 已用 / {torch.cuda.memory_reserved() / 1024**3:.2f}GB 预留"
def _format_ram():
    try:
        import psutil; p = psutil.Process(os.getpid()); vm = psutil.virtual_memory()
        return f"RAM: {p.memory_info().rss / 1024**3:.2f}GB RSS | 系统 {vm.used/1024**3:.2f}/{vm.total/1024**3:.2f}GB"
    except: return "RAM: (不可用)"

# 🔪 绞肉机：专治一切带有 LoRA 的幽灵克隆 Patcher，防止 VRAM/RAM 驻留
def _force_unload_patcher(patcher):
    if patcher is None: return
    try: patcher.unpatch_model(unpatch_weights=True)
    except: pass
    try: patcher.model_unload()
    except: pass
    if hasattr(mm, 'current_loaded_models'):
        # 防止对象克隆导致 id 不匹配，遍历移除
        to_remove =[m for m in mm.current_loaded_models if m is patcher or getattr(m, 'model', None) is getattr(patcher, 'model', object())]
        for m in to_remove:
            try: mm.current_loaded_models.remove(m)
            except: pass
    for attr in ('patches', 'backup', 'object_patches'):
        try: getattr(patcher, attr, {}).clear()
        except: pass
    _soft_vram_cleanup()

# 💥 核心防暴毙上下文管理器
@contextlib.contextmanager
def _vram_panic_mode(model_patcher, active=True):
    if not active or model_patcher is None:
        yield
        return

    import comfy.model_management as mm
    prev_state = getattr(mm, 'vram_state', None)
    if hasattr(mm, 'VRAMState'): mm.vram_state = mm.VRAMState.LOW_VRAM

    original_free = getattr(mm, 'get_free_memory', None)
    if original_free:
        def fake_free_memory(*args, **kwargs):
            res = original_free(*args, **kwargs)
            limit = 256 * 1024 * 1024
            if isinstance(res, tuple): return (min(res[0], limit), min(res[1], limit))
            return min(res, limit)
        mm.get_free_memory = fake_free_memory

    cls = model_patcher.__class__
    orig_load = getattr(cls, 'load', None)
    orig_patch = getattr(cls, 'patch_model', None)
    orig_partial = getattr(cls, 'partially_load', None)

    def safe_load(self, *args, **kwargs):
        kwargs.pop('force_patch_weights', None)
        if orig_load and 'force_patch_weights' in inspect.signature(orig_load).parameters: kwargs['force_patch_weights'] = False
        return orig_load(self, *args, **kwargs) if orig_load else None

    def safe_patch(self, *args, **kwargs):
        kwargs.pop('patch_weights', None)
        if orig_patch and 'patch_weights' in inspect.signature(orig_patch).parameters: kwargs['patch_weights'] = False
        return orig_patch(self, *args, **kwargs) if orig_patch else None

    def safe_partial(self, *args, **kwargs):
        kwargs.pop('force_patch_weights', None)
        if orig_partial and 'force_patch_weights' in inspect.signature(orig_partial).parameters: kwargs['force_patch_weights'] = False
        return orig_partial(self, *args, **kwargs) if orig_partial else None

    if orig_load: model_patcher.load = types.MethodType(safe_load, model_patcher)
    if orig_patch: model_patcher.patch_model = types.MethodType(safe_patch, model_patcher)
    if orig_partial: model_patcher.partially_load = types.MethodType(safe_partial, model_patcher)

    _soft_vram_cleanup()
    try:
        yield
    finally:
        if prev_state is not None: mm.vram_state = prev_state
        if original_free: mm.get_free_memory = original_free
        if orig_load:
            try: delattr(model_patcher, 'load')
            except: pass
        if orig_patch:
            try: delattr(model_patcher, 'patch_model')
            except: pass
        if orig_partial:
            try: delattr(model_patcher, 'partially_load')
            except: pass
        _soft_vram_cleanup()

# ==========================================
# 2. 智能阵列切片算法 (OOC)
# ==========================================
def get_optimal_tile_coords(w, h, tw, th, overlap=64):
    stride_x, stride_y = tw - overlap, th - overlap
    x_starts, y_starts, x, y = [],[], 0, 0
    while x < w:
        x_starts.append(x)
        if x + tw >= w:
            if x + tw > w: x_starts[-1] = max(0, w - tw)
            break
        x += stride_x
    while y < h:
        y_starts.append(y)
        if y + th >= h:
            if y + th > h: y_starts[-1] = max(0, h - th)
            break
        y += stride_y
    x_starts, y_starts = sorted(list(set(x_starts))), sorted(list(set(y_starts)))
    return[(x, y, min(x + tw, w), min(y + th, h)) for y in y_starts for x in x_starts]

def create_gradient_mask(w, h, direction="horizontal"):
    mask = Image.new("L", (w, h)); draw = ImageDraw.Draw(mask)
    if direction == "horizontal":
        for x in range(w): draw.line([(x, 0), (x, h)], fill=int((x / w) * 255))
    else:
        for y in range(h): draw.line([(0, y), (w, y)], fill=int((y / h) * 255))
    return mask

def blend_images(bg_img, fg_img, box, overlap=64):
    x, y, x2, y2 = box; tw, th = fg_img.size
    final_mask = Image.new("L", (tw, th), 255)
    if x > 0:
        final_mask.paste(create_gradient_mask(overlap, th, "horizontal"), (0, 0))
    if y > 0:
        grad_top = create_gradient_mask(tw, overlap, "vertical")
        full_top = Image.new("L", (tw, th), 255); full_top.paste(grad_top, (0, 0))
        final_mask = ImageChops.darker(final_mask, full_top)
    bg_img.paste(fg_img, (x, y), mask=final_mask)
    return bg_img

# ==========================================
# 3. 样式与组件
# ==========================================
STYLE_CSS = """<style>.zimg-panel { font-family: 'Segoe UI', sans-serif; }.zimg-section-title { color: #38bdf8; font-size: 15px; font-weight: 600; margin-bottom: 12px; padding-bottom: 8px; border-bottom: 1px solid #334155; }.zimg-header { background: linear-gradient(135deg, #0ea5e9 0%, #8b5cf6 50%, #ec4899 100%); border-radius: 16px; padding: 20px; margin-bottom: 16px; }.zimg-header h1 { color: white; font-size: 24px; font-weight: 700; margin: 0 0 4px 0; }.zimg-header p { color: rgba(255,255,255,0.85); font-size: 12px; margin: 0; }.zimg-label { color: #94a3b8; font-size: 12px; margin-bottom: 4px; }.zimg-info { color: #64748b; font-size: 11px; margin-top: 4px; }</style>"""
ASPECT_RATIOS = {"自定义": None, "1:1 方形 (1024×1024)": (1024, 1024), "4:3 横屏 (1152×864)": (1152, 864), "3:4 竖屏 (864×1152)": (864, 1152), "16:9 宽屏 (1344×768)": (1344, 768), "9:16 手机屏 (768×1344)": (768, 1344), "3:2 照片横 (1216×816)": (1216, 816), "2:3 照片竖 (816×1216)": (816, 1216)}

def make_section(title_html, *children): return widgets.VBox([widgets.HTML(f'<div class="zimg-section-title">{title_html}</div>'), *children], layout=widgets.Layout(padding="16px", margin="8px 0", border="1px solid #334155", border_radius="12px", background="#1e293b"))
def _row(*args, **kwargs): return widgets.HBox(list(args), layout=widgets.Layout(margin="5px 0", gap="10px", align_items="center", **kwargs))
def _col(*args): return widgets.VBox(list(args))

allocator_html = widgets.HTML(value=f'<div style="margin:0 0 10px 0; padding:10px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;"><div style="font-size:13px; color:#e2e8f0; font-weight:600; margin-bottom:4px;">🧠 Allocator 状态</div><div style="font-size:13px;">{_allocator_status_text()}</div></div>')
trans_mode_html = widgets.HTML(value='<div style="margin:0 0 10px 0; padding:8px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;"><span style="font-size:12px; color:#64748b; font-family:monospace;">🔧 Trans策略: 等待引擎初始化...</span></div>')
header_html = widgets.HTML(STYLE_CSS + '<div class="zimg-panel"><div class="zimg-header"><h1>🎨 Z-Image Turbo · v10.6</h1><p>按需切算防OOM · 绞肉机防RAM滞留死机 · 无损Lokr直吃</p></div></div>')

prompt_box = widgets.Textarea(value="一只猫咪坐在窗台上，阳光洒落，细节丰富，摄影质感", placeholder="输入正向提示词...", layout=widgets.Layout(width="100%", height="120px"))
neg_box = widgets.Textarea(value="blurry, ugly, bad, low quality, distorted, watermark", placeholder="输入负面提示词...", layout=widgets.Layout(width="100%", height="50px"))
w_png_meta_path = widgets.Text(value="", placeholder="输入 PNG 路径读取元数据", layout=widgets.Layout(flex="1"))
btn_load_meta = widgets.Button(description="📥 读取并回填", button_style="info", layout=widgets.Layout(width="120px"))
meta_status = widgets.HTML("<div class='zimg-info'>输入 PNG 路径可还原参数。</div>")

w_ratio = widgets.Dropdown(options=list(ASPECT_RATIOS.keys()), value="1:1 方形 (1024×1024)", layout=widgets.Layout(width="100%"))
w_width = widgets.IntSlider(value=1024, min=512, max=2560, step=16, description="宽度", style={'description_width': '40px'}, layout=widgets.Layout(flex="1"))
w_height = widgets.IntSlider(value=1024, min=512, max=2560, step=16, description="高度", style={'description_width': '40px'}, layout=widgets.Layout(flex="1"))
w_size_info = widgets.HTML('<div class="zimg-info">📐 最终尺寸: 1024 × 1024</div>')

_current_ratio, _updating = 1.0, False
def _on_ratio_change(c):
    global _current_ratio, _updating
    val = c["new"]
    if val == "自定义" or ASPECT_RATIOS[val] is None: _current_ratio = None
    else: w, h = ASPECT_RATIOS[val]; _current_ratio = w / h; _updating = True; w_width.value, w_height.value = w, h; _updating = False
    _update_size_info()
def _on_width_change(c):
    global _current_ratio, _updating
    if _updating or _current_ratio is None: return
    new_w = _round_to_16(c["new"]); new_h = max(512, min(2560, _round_to_16(new_w / _current_ratio)))
    _updating = True; w_height.value = new_h; _updating = False; _update_size_info()
def _on_height_change(c):
    global _current_ratio, _updating
    if _updating or _current_ratio is None: return
    new_h = _round_to_16(c["new"]); new_w = max(512, min(2560, _round_to_16(new_h * _current_ratio)))
    _updating = True; w_width.value = new_w; _updating = False; _update_size_info()
def _update_size_info(): w_size_info.value = f'<div class="zimg-info">📐 最终尺寸: {w_width.value} × {w_height.value} &nbsp;｜&nbsp; pixels: {w_width.value * w_height.value:,}</div>'
w_ratio.observe(_on_ratio_change, names="value"); w_width.observe(_on_width_change, names="value"); w_height.observe(_on_height_change, names="value")

w_steps = widgets.IntSlider(value=9, min=2, max=50, step=1, description="步数", style={'description_width': '60px'}, layout=widgets.Layout(flex="1"))
w_cfg = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description="CFG", style={'description_width': '40px'}, layout=widgets.Layout(flex="1"))
w_seed = widgets.IntText(value=0, description="种子(0随机)", style={'description_width': '80px'}, layout=widgets.Layout(flex="1"))
w_batch = widgets.BoundedIntText(value=1, min=1, max=10, step=1, description="每次张数", style={'description_width': '60px'}, layout=widgets.Layout(flex="1"))

_ALL_SAMPLERS =[
    "euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "lcm", "deis",
    "dpmpp_2m", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_2s_ancestral",
    "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu",
    "heun", "heun_3rd", "lms", "dpm_2", "dpm_2_ancestral", "dpm_fast", "dpm_adaptive",
    "ddpm", "ddim", "ipndm", "ipndm_v", "uni_pc", "uni_pc_bh2",
]
_ALL_SCHEDULERS =[
    "simple", "sgm_uniform", "karras", "exponential", "beta",
    "normal", "ddim_uniform", "linear_quadratic", "kl_optimal",
]

w_sampler = widgets.Dropdown(options=_ALL_SAMPLERS, value="euler", description="采样器", style={'description_width': '50px'}, layout=widgets.Layout(flex="1"))
w_scheduler = widgets.Dropdown(options=_ALL_SCHEDULERS, value="simple", description="调度器", style={'description_width': '50px'}, layout=widgets.Layout(flex="1"))

w_upscale_mode = widgets.Dropdown(
    options=[
        "1. 无 (按设定直出)",
        "2. Hires.fix (内置全图放大)",
        "3. Out-of-Core 极限硬盘切片重绘 (纯净Python拼接)",
        "4. Refiner (原生比例细节增强)",
    ],
    value="1. 无 (按设定直出)",
    description="放大策略:", style={'description_width': '70px'},
    layout=widgets.Layout(width="100%")
)
w_upscale_scale = widgets.FloatSlider(value=2.0, min=1.1, max=4.0, step=0.1, description="放大倍率", style={'description_width': '60px'}, layout=widgets.Layout(flex="1"))
w_upscale_denoise = widgets.FloatSlider(value=0.35, min=0.1, max=1.0, step=0.01, description="局部重绘", style={'description_width': '70px'}, layout=widgets.Layout(flex="1"))
w_ooc_grid = widgets.Dropdown(options=["4块 (2x2)", "9块 (3x3)", "16块 (4x4)"], value="4块 (2x2)", description="OOC阵列:", style={'description_width': '70px'}, layout=widgets.Layout(flex="1"))
w_ooc_upscaler = widgets.Dropdown(
    options=["4x-UltraSharp (写实/人像首选)", "RealESRGAN_x4plus (二次元首选)"],
    value="4x-UltraSharp (写实/人像首选)",
    description="OOC放大器:", style={'description_width': '70px'},
    layout=widgets.Layout(flex="1")
)

w_hires_upscaler = widgets.Dropdown(
    options=["潜空间 bislerp (最快, 细节少)", "潜空间 bicubic (较快, 细节少)", "像素空间 4x-UltraSharp (慢, 细节最佳)", "像素空间 ESRGAN (慢, 细节最佳)"],
    value="像素空间 4x-UltraSharp (慢, 细节最佳)", description="放大器:",
    style={'description_width': '55px'}, layout=widgets.Layout(width="100%")
)
w_hires_2nd_steps = widgets.IntSlider(value=0, min=0, max=50, step=1, description="步数(0跟随)", readout=True, continuous_update=False, style={'description_width': '75px'}, layout=widgets.Layout(flex="1"))
w_hires_2nd_cfg = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.1, description="CFG(0跟随)", readout=True, continuous_update=False, style={'description_width': '75px'}, layout=widgets.Layout(flex="1"))
w_hires_2nd_sampler = widgets.Dropdown(options=["跟随主采样器"] + _ALL_SAMPLERS, value="跟随主采样器", description="二阶段采样", style={'description_width': '80px'}, layout=widgets.Layout(flex="1"))
w_hires_2nd_scheduler = widgets.Dropdown(options=["跟随主调度器"] + _ALL_SCHEDULERS, value="跟随主调度器", description="二阶段调度", style={'description_width': '80px'}, layout=widgets.Layout(flex="1"))
w_hires_noise_aug = widgets.FloatSlider(value=0.03, min=0.0, max=0.3, step=0.005, description="噪声注入", style={'description_width': '65px'}, layout=widgets.Layout(flex="1"))
w_hires_latent_sharpen = widgets.FloatSlider(value=0.15, min=0.0, max=0.6, step=0.01, description="潜空间锐化", style={'description_width': '80px'}, layout=widgets.Layout(flex="1"))
w_hires_blend = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description="原始混合", style={'description_width': '65px'}, layout=widgets.Layout(flex="1"))

hires_extra = widgets.VBox([
    w_hires_upscaler,
    _row(w_hires_2nd_steps, w_hires_2nd_cfg, w_hires_2nd_sampler, w_hires_2nd_scheduler),
    _row(w_hires_noise_aug, w_hires_latent_sharpen, w_hires_blend),
    widgets.HTML('<div class="zimg-info">💡 像素空间放大 = 解码→ESRGAN放大→重编码→精修，细节远超纯潜空间放大。<br>🔁 Refiner 模式 = 直接在原生 Latent 上锐化+注噪+二次采样，无 VAE 往返损耗，推荐采样器：dpmpp_2m_sde / dpmpp_3m_sde + karras。</div>'),
], layout=widgets.Layout(display="none", flex_flow="column", gap="4px"))

def _on_upscale_mode_change(c):
    is_hires   = "Hires"   in c["new"]
    is_refiner = "Refiner" in c["new"]
    hires_extra.layout.display = "flex" if (is_hires or is_refiner) else "none"
    w_hires_upscaler.disabled = is_refiner
    w_upscale_scale.disabled  = is_refiner
    w_ooc_upscaler.disabled   = is_refiner
    w_ooc_grid.disabled       = is_refiner
w_upscale_mode.observe(_on_upscale_mode_change, names="value")

LORA_DIR = "/content/ComfyUI/models/loras"; os.makedirs(LORA_DIR, exist_ok=True)
def _get_lora_files(): return sorted([f for f in os.listdir(LORA_DIR) if f.endswith(('.safetensors', '.pt', '.ckpt'))]) if os.path.exists(LORA_DIR) else[]
def _make_lora_slot(idx):
    opts =[("（不加载）", "")] +[(f, f) for f in _get_lora_files()]
    return (widgets.Dropdown(options=opts, value="", description=f"槽{idx}", style={'description_width': '30px'}, layout=widgets.Layout(width="40%")), widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.05, description="模型强度", style={'description_width': '60px'}, layout=widgets.Layout(width="30%")), widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.05, description="CLIP强度", style={'description_width': '60px'}, layout=widgets.Layout(width="30%")))
lora_dd_1, lora_sm_1, lora_sc_1 = _make_lora_slot(1); lora_dd_2, lora_sm_2, lora_sc_2 = _make_lora_slot(2); lora_dd_3, lora_sm_3, lora_sc_3 = _make_lora_slot(3)
btn_refresh_lora = widgets.Button(description="🔄 刷新", button_style="info", layout=widgets.Layout(width="80px"))

def _on_refresh_lora(_):
    opts =[("（不加载）", "")] +[(f, f) for f in _get_lora_files()]
    for dd in[lora_dd_1, lora_dd_2, lora_dd_3]:
        cur = dd.value; dd.options = opts; dd.value = cur if cur in [o[1] for o in opts] else ""
btn_refresh_lora.on_click(_on_refresh_lora)

# ==========================================
# ★ LoRA 相位分离工具函数（带扰民屏蔽器）
# ==========================================
@contextlib.contextmanager
def suppress_lora_warnings():
    import logging
    logger = logging.getLogger()
    class LoraWarningFilter(logging.Filter):
        def filter(self, record):
            return "lora key not loaded" not in record.getMessage()
    f = LoraWarningFilter()
    logger.addFilter(f)
    try:
        yield
    finally:
        logger.removeFilter(f)

def _load_lora_raw(lora_name: str) -> dict:
    import comfy.utils
    try:
        import folder_paths
        path = folder_paths.get_full_path("loras", lora_name)
    except Exception:
        path = None
    if path is None: path = os.path.join(LORA_DIR, lora_name)
    if not os.path.exists(path): raise FileNotFoundError(f"LoRA 文件未找到: {lora_name}")
    return comfy.utils.load_torch_file(path, safe_load=True)

def _apply_clip_lora(clip, lora_name: str, strength_clip: float):
    import comfy.sd
    if abs(strength_clip) < 1e-6:
        print(f"    ⏭️ [{lora_name}] CLIP强度=0，跳过 Phase-1")
        return clip
    raw = _load_lora_raw(lora_name)
    with suppress_lora_warnings():
        _, new_clip = comfy.sd.load_lora_for_models(None, clip, raw, 0.0, strength_clip)
    del raw
    print(f"    ✅ {lora_name}  clip={strength_clip:.2f}")
    return new_clip

def _apply_unet_lora(model, lora_name: str, strength_model: float):
    import comfy.sd
    if abs(strength_model) < 1e-6:
        print(f"    ⏭️[{lora_name}] 模型强度=0，跳过 Phase-2")
        return model
    raw = _load_lora_raw(lora_name)
    with suppress_lora_warnings():
        new_model, _ = comfy.sd.load_lora_for_models(model, None, raw, strength_model, 0.0)
    del raw
    print(f"    ✅ {lora_name}  model={strength_model:.2f}")
    return new_model

# ==========================================
# 保存设置
# ==========================================
w_save_custom = widgets.Checkbox(value=False, description="保存到指定目录", layout=widgets.Layout(width="100%"))
w_custom_dir = widgets.Text(value="/content/drive/MyDrive/Z-image/Outputs", placeholder="输入保存路径", layout=widgets.Layout(width="100%")); w_custom_dir.disabled = True
def _on_save_custom_change(c): w_custom_dir.disabled = not c["new"]
w_save_custom.observe(_on_save_custom_change, names="value")

w_vae_tiled = widgets.Checkbox(value=False, description="基础图分块解码 (Tiled VAE) ← T4建议开启", layout=widgets.Layout(width="100%"))
w_tile_size = widgets.IntSlider(value=320, min=192, max=1024, step=64, description="块大小 px", style={'description_width': '70px'}, layout=widgets.Layout(width="100%")); w_tile_size.disabled = True
def _on_tiled_change(c): w_tile_size.disabled = not c["new"]
w_vae_tiled.observe(_on_tiled_change, names="value")

# ==========================================
# SVE (SeedVarianceEnhancer)
# ==========================================
w_svh_enable = widgets.Checkbox(value=False, description="启用 SeedVarianceEnhancer (变异增强)", indent=False, layout=widgets.Layout(width="100%"))
w_svh_randomize_percent = widgets.FloatSlider(value=50.0, min=1.0, max=100.0, step=1.0, description="randomize%", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_strength = widgets.FloatText(value=20.0, description="strength", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_noise_insert = widgets.Dropdown(options=["noise on beginning steps", "noise on ending steps", "noise on all steps", "disabled"], value="noise on beginning steps", description="noise_insert", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_steps_switchover_percent = widgets.FloatSlider(value=20.0, min=1.0, max=99.0, step=1.0, description="switch%", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_seed_mode = widgets.Dropdown(options=[("跟随出图种子", "follow"), ("自定义 SVH 种子", "custom")], value="follow", description="SVH seed", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_seed = widgets.IntText(value=0, description="seed", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_mask_starts_at = widgets.Dropdown(options=["beginning", "end"], value="beginning", description="mask_from", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_mask_percent = widgets.FloatSlider(value=0.0, min=0.0, max=99.0, step=1.0, description="mask%", style={'description_width': '110px'}, layout=widgets.Layout(flex="1"))
w_svh_log_to_console = widgets.Checkbox(value=False, description="log_to_console", indent=False, layout=widgets.Layout(width="100%"))

def _svh_controls_enabled(enable: bool):
    for w in[w_svh_randomize_percent, w_svh_strength, w_svh_noise_insert, w_svh_steps_switchover_percent, w_svh_seed_mode, w_svh_seed, w_svh_mask_starts_at, w_svh_mask_percent, w_svh_log_to_console]:
        w.disabled = not bool(enable)

def _on_svh_enable_change(change):
    _svh_controls_enabled(change["new"])
    _on_svh_seed_mode_change({"new": w_svh_seed_mode.value})
    sve_controls.layout.display = "flex" if change["new"] else "none"

def _on_svh_seed_mode_change(change):
    w_svh_seed.disabled = (change["new"] != "custom") or (not bool(w_svh_enable.value))

w_svh_enable.observe(_on_svh_enable_change, names="value")
w_svh_seed_mode.observe(_on_svh_seed_mode_change, names="value")

sve_controls = widgets.VBox([
    _row(w_svh_randomize_percent, w_svh_strength),
    _row(w_svh_noise_insert, w_svh_steps_switchover_percent),
    _row(w_svh_mask_starts_at, w_svh_mask_percent),
    _row(w_svh_seed_mode, w_svh_seed),
    w_svh_log_to_console
], layout=widgets.Layout(display="none", flex_flow="column", gap="4px"))

_svh_controls_enabled(w_svh_enable.value)
_on_svh_seed_mode_change({"new": w_svh_seed_mode.value})

# ==========================================
# 4. 智能画板系统 (Inpainting)
# ==========================================
_inpaint_src_pil = None
_inpaint_mask_np = None
_inpaint_ready = False

w_inpaint_enable = widgets.Checkbox(value=False, description="🖌️ 启用局部重绘 (Inpainting)", layout=widgets.Layout(width="100%"))
w_inpaint_upload = widgets.FileUpload(accept='image/*', multiple=False, description='📁 本地上传', button_style='primary', layout=widgets.Layout(width="120px"))
w_inpaint_path = widgets.Text(placeholder="或输入云端图片路径加载...", layout=widgets.Layout(flex="1"))
btn_inpaint_path = widgets.Button(description="📂 路径加载", button_style="info", layout=widgets.Layout(width="100px"))
w_inpaint_denoise = widgets.FloatSlider(value=0.75, min=0.01, max=1.0, step=0.01, description="重绘强度", style={'description_width': '60px'}, layout=widgets.Layout(flex="1"))
w_inpaint_mask_blur = widgets.IntSlider(value=8, min=0, max=64, step=1, description="边缘模糊", style={'description_width': '60px'}, layout=widgets.Layout(flex="1"))
w_inpaint_mask_mode = widgets.Dropdown(options=["重绘涂抹区域", "保留涂抹区域"], value="重绘涂抹区域", description="模式", style={'description_width': '40px'}, layout=widgets.Layout(width="150px"))
w_inpaint_status = widgets.HTML('<div class="zimg-info" style="color:#f59e0b;">请上传或输入路径加载底图...</div>')
painter_out = widgets.Output(layout=widgets.Layout(border="1px dashed #475569", margin="8px 0", padding="8px", background="#0b1220"))

def receive_mask(b64_str):
    global _inpaint_mask_np, _inpaint_ready
    try:
        header, encoded = b64_str.split(",", 1)
        data = base64.b64decode(encoded)
        mask_pil = Image.open(io.BytesIO(data)).convert("L")
        _inpaint_mask_np = np.array(mask_pil, dtype=np.float32) / 255.0
        _inpaint_ready = True
        w_inpaint_status.value = f'<div style="color:#22c55e;font-weight:bold;">✅ 遮罩已保存并确认！面积: {_inpaint_mask_np.mean()*100:.1f}%</div>'
    except Exception as e:
        w_inpaint_status.value = f'<div style="color:#ef4444;">❌ 遮罩保存失败: {e}</div>'
if 'google.colab' in sys.modules: output.register_callback('zimg_save_mask', receive_mask)

def render_inpaint_board():
    global _inpaint_src_pil, _inpaint_ready, _inpaint_mask_np
    if _inpaint_src_pil is None: return
    _inpaint_ready = False; _inpaint_mask_np = None
    w_inpaint_status.value = '<div class="zimg-info" style="color:#eab308;">✅ 图片已加载，请在下方画板涂抹后点击【✅ 确认并应用遮罩】</div>'
    w, h = _inpaint_src_pil.size
    scale = min(760.0 / w, 1.0); dw, dh = int(w * scale), int(h * scale)
    buf = io.BytesIO(); _inpaint_src_pil.resize((dw, dh), Image.LANCZOS).save(buf, format='JPEG')
    b64_img = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode('utf-8')
    html_code = f"""
    <div style="margin-bottom:8px; display:flex; gap:10px; align-items:center;">
        <span style="color:#e2e8f0; font-size:13px;">画笔粗细:</span>
        <input type="range" id="bSize" min="5" max="100" value="30" style="width:100px;">
        <label style="color:#e2e8f0; font-size:13px; cursor:pointer;"><input type="checkbox" id="bEraser"> 橡皮擦</label>
        <button onclick="clearMask()" style="background:#ef4444; color:white; border:none; padding:4px 10px; border-radius:4px;">🗑️ 清空</button>
        <button id="btnConfirm" onclick="sendMaskToColab()" style="background:#22c55e; color:white; border:none; padding:4px 10px; border-radius:4px; font-weight:bold;">✅ 确认并应用遮罩</button>
    </div>
    <div style="position:relative; width:{dw}px; height:{dh}px; touch-action:none; cursor:none; overflow:hidden;" id="canvasWrap">
        <img src="{b64_img}" style="position:absolute; top:0; left:0; width:{dw}px; height:{dh}px; pointer-events:none;">
        <canvas id="maskLayer" width="{dw}" height="{dh}" style="position:absolute; top:0; left:0; opacity:0.6; mix-blend-mode:normal;"></canvas>
        <div id="brushCursor" style="position:absolute; border-radius:50%; border:2px solid rgba(255,255,255,0.8); background:rgba(255,0,0,0.3); pointer-events:none; transform:translate(-50%, -50%); display:none;"></div>
    </div>
    <script>
        var canvas = document.getElementById('maskLayer'); var ctx = canvas.getContext('2d');
        var wrap = document.getElementById('canvasWrap'); var cursor = document.getElementById('brushCursor');
        var bSize = document.getElementById('bSize'); var bEraser = document.getElementById('bEraser');
        var isDrawing = false; ctx.lineJoin = 'round'; ctx.lineCap = 'round';
        function getPos(e) {{ var rect = canvas.getBoundingClientRect(); var x = e.clientX ? e.clientX : e.touches[0].clientX; var y = e.clientY ? e.clientY : e.touches[0].clientY; return {{ x: x - rect.left, y: y - rect.top }}; }}
        function updateCursor(pos) {{ cursor.style.width = bSize.value + 'px'; cursor.style.height = bSize.value + 'px'; cursor.style.left = pos.x + 'px'; cursor.style.top = pos.y + 'px'; cursor.style.background = bEraser.checked ? 'rgba(0,0,0,0.4)' : 'rgba(255,0,0,0.3)'; }}
        wrap.addEventListener('mouseenter', () => cursor.style.display = 'block');
        wrap.addEventListener('mouseleave', () => {{ cursor.style.display = 'none'; isDrawing = false; }});
        wrap.addEventListener('mousemove', (e) => updateCursor(getPos(e)));
        canvas.addEventListener('mousedown', (e) => {{ e.preventDefault(); isDrawing = true; var pos = getPos(e); ctx.beginPath(); ctx.moveTo(pos.x, pos.y); updateCursor(pos); }});
        canvas.addEventListener('mousemove', (e) => {{ if (!isDrawing) return; e.preventDefault(); var pos = getPos(e); ctx.lineWidth = bSize.value; ctx.globalCompositeOperation = bEraser.checked ? 'destination-out' : 'source-over'; ctx.strokeStyle = bEraser.checked ? 'rgba(0,0,0,1)' : 'rgba(255,0,0,1)'; ctx.lineTo(pos.x, pos.y); ctx.stroke(); updateCursor(pos); }});
        canvas.addEventListener('mouseup', () => isDrawing = false);
        window.clearMask = function() {{ ctx.clearRect(0, 0, canvas.width, canvas.height); }}
        window.sendMaskToColab = function() {{
            var btn = document.getElementById('btnConfirm'); btn.innerText = '⏳ 同步中...'; btn.style.background = '#eab308';
            var offCanvas = document.createElement('canvas'); offCanvas.width = canvas.width; offCanvas.height = canvas.height;
            var offCtx = offCanvas.getContext('2d'); offCtx.fillStyle = 'black'; offCtx.fillRect(0, 0, offCanvas.width, offCanvas.height);
            var imgData = ctx.getImageData(0, 0, canvas.width, canvas.height); var outData = offCtx.createImageData(canvas.width, canvas.height);
            for(var i=0; i<imgData.data.length; i+=4) {{ if(imgData.data[i+3]>0) {{ outData.data[i]=outData.data[i+1]=outData.data[i+2]=outData.data[i+3]=255; }} else outData.data[i+3]=255; }}
            offCtx.putImageData(outData, 0, 0);
            google.colab.kernel.invokeFunction('zimg_save_mask',[offCanvas.toDataURL('image/png')], {{}});
            setTimeout(() => {{ btn.innerText = '✅ 已应用'; btn.style.background = '#22c55e'; setTimeout(()=>btn.innerText='✅ 确认并应用遮罩', 2000); }}, 500);
        }}
    </script>
    """
    with painter_out: clear_output(wait=True); display(HTML(html_code))

def _on_upload_change(c):
    global _inpaint_src_pil
    v = c['owner'].value
    if not v: return
    data = v[list(v.keys())[0]]['content'] if isinstance(v, dict) else v[0]['content']
    _inpaint_src_pil = Image.open(io.BytesIO(data)).convert("RGB")
    render_inpaint_board()
w_inpaint_upload.observe(_on_upload_change, names='value')

def _on_load_path(_):
    global _inpaint_src_pil
    path = w_inpaint_path.value.strip()
    if os.path.exists(path):
        _inpaint_src_pil = Image.open(path).convert("RGB")
        render_inpaint_board()
    else: w_inpaint_status.value = '<div class="zimg-info" style="color:#ef4444;">❌ 路径不存在，请检查</div>'
btn_inpaint_path.on_click(_on_load_path)

inpaint_box = widgets.VBox([_row(w_inpaint_upload, w_inpaint_path, btn_inpaint_path), _row(w_inpaint_denoise, w_inpaint_mask_blur, w_inpaint_mask_mode), painter_out, w_inpaint_status], layout=widgets.Layout(visibility="hidden", margin="10px 0", padding="10px", border="1px solid #475569", border_radius="8px"))

def _on_inp_toggle(c):
    inpaint_box.layout.visibility = "visible" if c["new"] else "hidden"
    if c["new"] and _inpaint_src_pil is not None and not _inpaint_ready: render_inpaint_board()
w_inpaint_enable.observe(_on_inp_toggle, names="value")

def _on_load_meta(_):
    png_path = w_png_meta_path.value.strip()
    if not png_path or not os.path.exists(png_path): meta_status.value = "<div class='zimg-info' style='color:#ef4444;'>❌ 未找到文件</div>"; return
    try:
        img = Image.open(png_path)
        if hasattr(img, "info") and "parameters" in img.info:
            meta = json.loads(img.info["parameters"])
            if "prompt" in meta: prompt_box.value = meta["prompt"]
            if "negative_prompt" in meta: neg_box.value = meta["negative_prompt"]
            if "steps" in meta: w_steps.value = int(meta["steps"])
            if "cfg_scale" in meta: w_cfg.value = float(meta["cfg_scale"])
            if "seed" in meta: w_seed.value = int(meta["seed"])
            if "sampler_name" in meta:
                v = meta["sampler_name"]
                if v in _ALL_SAMPLERS: w_sampler.value = v
            if "scheduler" in meta:
                v = meta["scheduler"]
                if v in _ALL_SCHEDULERS: w_scheduler.value = v
            if meta.get("width") and meta.get("height"): w_ratio.value = "自定义"; w_width.value = int(meta["width"]); w_height.value = int(meta["height"])
            if "vae_tiled" in meta: w_vae_tiled.value = bool(meta["vae_tiled"])
            if "upscale_mode" in meta:
                v = meta["upscale_mode"]
                if v in[o for o in w_upscale_mode.options]: w_upscale_mode.value = v
            if "upscale_scale" in meta: w_upscale_scale.value = float(meta["upscale_scale"])
            if "upscale_denoise" in meta: w_upscale_denoise.value = float(meta["upscale_denoise"])
            if "upscale_grid" in meta: w_ooc_grid.value = meta["upscale_grid"]
            if "ooc_upscaler" in meta:
                v = meta["ooc_upscaler"]
                if v in[o for o in w_ooc_upscaler.options]: w_ooc_upscaler.value = v
            if "hires_upscaler" in meta:
                v = meta["hires_upscaler"]
                if v in[o for o in w_hires_upscaler.options]: w_hires_upscaler.value = v
            if "hires_2nd_steps" in meta: w_hires_2nd_steps.value = int(meta["hires_2nd_steps"])
            if "hires_2nd_cfg" in meta: w_hires_2nd_cfg.value = float(meta["hires_2nd_cfg"])
            if "hires_2nd_sampler" in meta:
                v = meta["hires_2nd_sampler"]
                if v in[o for o in w_hires_2nd_sampler.options]: w_hires_2nd_sampler.value = v
            if "hires_2nd_scheduler" in meta:
                v = meta["hires_2nd_scheduler"]
                if v in[o for o in w_hires_2nd_scheduler.options]: w_hires_2nd_scheduler.value = v
            if "hires_noise_aug" in meta: w_hires_noise_aug.value = float(meta["hires_noise_aug"])
            if "hires_latent_sharpen" in meta: w_hires_latent_sharpen.value = float(meta["hires_latent_sharpen"])
            if "hires_blend" in meta: w_hires_blend.value = float(meta["hires_blend"])
            if "seed_variance_enhancer" in meta:
                svh = meta["seed_variance_enhancer"]
                w_svh_enable.value = bool(svh.get("enabled", False))
                w_svh_randomize_percent.value = float(svh.get("randomize_percent", 50.0))
                w_svh_strength.value = float(svh.get("strength", 20.0))
                ni = svh.get("noise_insert", "noise on all steps")
                if ni in[o for o in w_svh_noise_insert.options]: w_svh_noise_insert.value = ni
                w_svh_steps_switchover_percent.value = float(svh.get("steps_switchover_percent", 20.0))
                sm = svh.get("seed_mode", "follow")
                if sm in[v for _, v in w_svh_seed_mode.options]: w_svh_seed_mode.value = sm
                w_svh_seed.value = int(svh.get("seed", 0))
                ms = svh.get("mask_starts_at", "beginning")
                if ms in[o for o in w_svh_mask_starts_at.options]: w_svh_mask_starts_at.value = ms
                w_svh_mask_percent.value = float(svh.get("mask_percent", 0.0))
                w_svh_log_to_console.value = bool(svh.get("log_to_console", False))
            meta_status.value = f"<div class='zimg-info' style='color:#22c55e;'>✅ 成功还原：{os.path.basename(png_path)}</div>"
    except Exception as e: meta_status.value = f"<div class='zimg-info' style='color:#ef4444;'>❌ 错误: {e}</div>"
btn_load_meta.on_click(_on_load_meta)

# ==========================================
# 5. UI清理与全量卸载
# ==========================================
out = widgets.Output(layout=widgets.Layout(border="2px solid #334155", border_radius="12px", padding="16px", min_height="200px", background="#0b1220"))

def _on_full_unload(_):
    out.clear_output()
    with out:
        print("⛔ 开始执行深度完整卸载...")
        _engine = getattr(__main__, '_engine', None)
        if _engine is not None:
            freed = _engine.destroy()
            print(f"✅ 电焊引擎已销毁: {', '.join(freed) if freed else '（无存留）'}")
            __main__._engine = None
        else:
            print("⚠️ 未检测到运行中的引擎。")
        for loaded in list(mm.current_loaded_models):
            try: loaded.model_unload()
            except: pass
        mm.current_loaded_models.clear()
        gc.collect(); gc.collect(); gc.collect()
        _soft_vram_cleanup()
        try: ctypes.CDLL("libc.so.6").malloc_trim(0)
        except: pass
        print(f"✅ VRAM/RAM 已强行回收。")
        print(_format_vram()); print(_format_ram())

btn_run = widgets.Button(description="🚀 开始生成图像", button_style="success", layout=widgets.Layout(width="200px", height="50px"))
btn_clear = widgets.Button(description="🗑️ 清空输出台", button_style="warning", layout=widgets.Layout(width="110px", height="50px"))
btn_full_unload = widgets.Button(description="⛔ 彻底卸载模型", button_style="danger", layout=widgets.Layout(width="130px", height="50px"))

# ==========================================
# 6. ★ 核心生成函数（RAM 终极防溢出版）
# ==========================================
def run_generation(_):
    out.clear_output()
    with out:
        _engine = getattr(__main__, '_engine', None)

        if _engine is None:
            print("❌ 错误: 未找到引擎实例！请先在【模型管理器】中执行加载。"); return
        on_demand = getattr(_engine, '_trans_on_demand', False)
        if not on_demand and getattr(_engine, 'model', None) is None:
            print("❌ 错误: Trans 未加载！请重新执行【电焊加载】。"); return
        if getattr(_engine, 'vae', None) is None:
            print("❌ 错误: VAE 未加载！请重新执行加载。"); return

        if on_demand:
            trans_mode_html.value = '<div style="margin:0 0 10px 0; padding:8px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;"><span style="font-size:12px; color:#f59e0b; font-weight:600; font-family:monospace;">♻️ Trans: 串行按需 · TE独占→CLIP-LoRA→编码→销毁→Trans独占→UNet-LoRA→采样</span></div>'
        else:
            trans_mode_html.value = '<div style="margin:0 0 10px 0; padding:8px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;"><span style="font-size:12px; color:#22c55e; font-weight:600; font-family:monospace;">⚡ Trans: 焊死常驻 · 标准模式</span></div>'

        te_name = getattr(_engine, '_default_te', None)
        if not te_name:
            print("❌ 错误: 引擎缺少 TE 文件名记录，请重新执行加载。"); return

        is_inp = w_inpaint_enable.value
        if is_inp:
            if _inpaint_src_pil is None: print("❌ 请先上传或加载底图！"); return
            if not _inpaint_ready or _inpaint_mask_np is None: print("❌ 请在画板点击【确认并应用遮罩】！"); return

        width16, height16 = _round_to_16(w_width.value), _round_to_16(w_height.value)
        steps, cfg, base_seed, batch_n = w_steps.value, w_cfg.value, w_seed.value, w_batch.value
        sampler, sched = w_sampler.value, w_scheduler.value
        use_tiled, tile_size = w_vae_tiled.value, w_tile_size.value
        prompt_str, neg_str = prompt_box.value, neg_box.value
        upscale_mode, scale_val, denoise_val = w_upscale_mode.value, w_upscale_scale.value, w_upscale_denoise.value
        cur_denoise = w_inpaint_denoise.value if is_inp else 1.0

        lora_slots =[
            (lora_dd_1.value, lora_sm_1.value, lora_sc_1.value),
            (lora_dd_2.value, lora_sm_2.value, lora_sc_2.value),
            (lora_dd_3.value, lora_sm_3.value, lora_sc_3.value),
        ]
        active_loras =[(n, sm, sc) for n, sm, sc in lora_slots if n]

        out_dir = w_custom_dir.value.strip() if w_save_custom.value else "/content/ComfyUI/output"
        os.makedirs(out_dir, exist_ok=True)

        print("=" * 60)
        print(f"🚀 开始生成  {width16}×{height16}  共 {batch_n} 张")
        print(f"   放大策略: {upscale_mode.split(' ')[1]}")
        print(f"   Trans模式: {'♻️ 串行按需 [LoRA相位分离]' if on_demand else '⚡ 焊死常驻'}")
        if active_loras: print(f"   LoRA 槽位: {[n for n,_,_ in active_loras]}")

        for i in range(batch_n):
            real_seed = (base_seed + i if base_seed != 0 else random.randint(1, 2 ** 62))
            print(f"\n{'─'*50}\n🖼️[{i+1}/{batch_n}]  seed={real_seed}")

            run_model = run_clip = run_vae = None
            positive = negative = latent_image = samples = decoded_cpu_batch = img = final_img = None
            base_image_tensor = upscaled_4x = target_tensor = target_pil = None

            try:
                run_vae = getattr(_engine, 'vae')
                CLIPTextEncode  = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
                EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
                KSampler        = NODE_CLASS_MAPPINGS["KSampler"]()
                VAEDecode       = NODE_CLASS_MAPPINGS["VAEDecode"]()
                VAEDecodeTiled  = NODE_CLASS_MAPPINGS["VAEDecodeTiled"]()

                # ══════════════════════════════════════════════════════
                # ★ 分支 A：串行按需模式 — LoRA 相位严格分离
                # ══════════════════════════════════════════════════════
                if on_demand:

                    # ── Phase-1: TE 独占 ──────────────────────────────
                    print(f"  📥[Phase-1 · TE独占] 加载 Text Encoder: {te_name}")
                    _engine._load_te(te_name)
                    run_clip = getattr(_engine, 'clip')
                    print(f"  ✅ TE 就绪 [{_format_vram()}]")

                    if active_loras:
                        print(f"  🎭 Phase-1 CLIP-LoRA 注入 ({len(active_loras)} 槽):")
                        for l_name, s_m, s_c in active_loras:
                            run_clip = _apply_clip_lora(run_clip, l_name, s_c)

                    print(f"  ⏳[Phase-1] 文本编码中...[{_format_ram()}]")
                    with torch.inference_mode():
                        positive_tmp = CLIPTextEncode.encode(run_clip, prompt_str)[0]
                        negative_tmp = CLIPTextEncode.encode(run_clip, neg_str)[0]

                    svh_meta = {}
                    if w_svh_enable.value:
                        svh_seed = int(w_svh_seed.value) if w_svh_seed_mode.value == "custom" else real_seed
                        svh_meta = {
                            "enabled": True, "randomize_percent": float(w_svh_randomize_percent.value),
                            "strength": float(w_svh_strength.value), "noise_insert": w_svh_noise_insert.value,
                            "steps_switchover_percent": float(w_svh_steps_switchover_percent.value),
                            "seed_mode": w_svh_seed_mode.value, "seed": svh_seed,
                            "mask_starts_at": w_svh_mask_starts_at.value, "mask_percent": float(w_svh_mask_percent.value),
                            "log_to_console": w_svh_log_to_console.value
                        }
                        try:
                            SVE_Class = NODE_CLASS_MAPPINGS.get("SeedVarianceEnhancer")
                            if SVE_Class:
                                sve_node = SVE_Class()
                                func_name = getattr(SVE_Class, "FUNCTION", "enhance")
                                sve_kwargs = svh_meta.copy(); sve_kwargs.pop("enabled"); sve_kwargs.pop("seed_mode")
                                sve_kwargs["conditioning"] = positive_tmp
                                positive_tmp = getattr(sve_node, func_name)(**sve_kwargs)[0]
                                print(f"  ✨ SVE 已作用: seed={svh_seed}, strength={w_svh_strength.value}")
                        except Exception as e: print(f"  ⚠️ SVE 应用失败: {e}")

                    positive = _engine._detach_cond(positive_tmp)
                    negative = _engine._detach_cond(negative_tmp)
                    del positive_tmp, negative_tmp

                    # 🔪 极致清理1：剥离幽灵克隆 Patcher 并回收 RAM
                    _force_unload_patcher(run_clip)
                    del run_clip; run_clip = None
                    _engine._destroy_te()
                    gc.collect(); gc.collect(); _soft_vram_cleanup()
                    print(f"  🔪[Phase-1 完成] TE 已彻底销毁，VRAM 归零 [{_format_vram()}]")
                    print(f"  🧹 [RAM状态] {_format_ram()}")

                    # ── Phase-2: Trans 独占 ───────────────────────────
                    print(f"  📥[Phase-2 · Trans独占] 加载 Transformer: {_engine._unet_name}")
                    _engine._stream_load_transformer()
                    run_model = getattr(_engine, 'model')
                    print(f"  ✅ Transformer 就绪[{_format_vram()}]")

                    if active_loras:
                        print(f"  🎭 Phase-2 UNet-LoRA 注入 ({len(active_loras)} 槽):")
                        for l_name, s_m, s_c in active_loras:
                            run_model = _apply_unet_lora(run_model, l_name, s_m)
                        # 清理读取 LoRA 遗留的内存
                        gc.collect(); _soft_vram_cleanup()

                # ══════════════════════════════════════════════════════
                # ★ 分支 B：电焊常驻模式 — Trans+TE 同时在场
                # ══════════════════════════════════════════════════════
                else:
                    print(f"  📥[即抛模式] 拉起 Text Encoder: {te_name}")
                    _engine._load_te(te_name)
                    run_model = getattr(_engine, 'model')
                    run_clip  = getattr(_engine, 'clip')

                    if active_loras:
                        LoraLoader = NODE_CLASS_MAPPINGS["LoraLoader"]()
                        print(f"  🎭 全量 LoRA 注入 ({len(active_loras)} 槽):")
                        for l_name, s_m, s_c in active_loras:
                            run_model, run_clip = LoraLoader.load_lora(run_model, run_clip, l_name, s_m, s_c)
                            print(f"    ✅ {l_name}  model={s_m:.2f}  clip={s_c:.2f}")

                    print(f"  ⏳ 文本编码中... [{_format_ram()}]")
                    with torch.inference_mode():
                        positive_tmp = CLIPTextEncode.encode(run_clip, prompt_str)[0]
                        negative_tmp = CLIPTextEncode.encode(run_clip, neg_str)[0]

                    svh_meta = {}
                    if w_svh_enable.value:
                        svh_seed = int(w_svh_seed.value) if w_svh_seed_mode.value == "custom" else real_seed
                        svh_meta = {
                            "enabled": True, "randomize_percent": float(w_svh_randomize_percent.value),
                            "strength": float(w_svh_strength.value), "noise_insert": w_svh_noise_insert.value,
                            "steps_switchover_percent": float(w_svh_steps_switchover_percent.value),
                            "seed_mode": w_svh_seed_mode.value, "seed": svh_seed,
                            "mask_starts_at": w_svh_mask_starts_at.value, "mask_percent": float(w_svh_mask_percent.value),
                            "log_to_console": w_svh_log_to_console.value
                        }
                        try:
                            SVE_Class = NODE_CLASS_MAPPINGS.get("SeedVarianceEnhancer")
                            if SVE_Class:
                                sve_node = SVE_Class()
                                func_name = getattr(SVE_Class, "FUNCTION", "enhance")
                                sve_kwargs = svh_meta.copy(); sve_kwargs.pop("enabled"); sve_kwargs.pop("seed_mode")
                                sve_kwargs["conditioning"] = positive_tmp
                                positive_tmp = getattr(sve_node, func_name)(**sve_kwargs)[0]
                        except Exception as e: pass

                    positive = _engine._detach_cond(positive_tmp)
                    negative = _engine._detach_cond(negative_tmp)
                    del positive_tmp, negative_tmp

                    # 🔪 极致清理1：剥离可能存在的 TE 幽灵
                    _force_unload_patcher(run_clip)
                    del run_clip; run_clip = None
                    _engine._destroy_te()
                    gc.collect(); gc.collect(); _soft_vram_cleanup()
                    print(f"  🔪 TE 已彻底销毁 [{_format_vram()}] | {_format_ram()}")

                # ══════════════════════════════════════════════════════
                # Inpainting 准备
                # ══════════════════════════════════════════════════════
                if is_inp:
                    VAEEncode_node = NODE_CLASS_MAPPINGS["VAEEncode"]()
                    SetLatentNoiseMask_node = NODE_CLASS_MAPPINGS["SetLatentNoiseMask"]()
                    base_resized = _inpaint_src_pil.resize((width16, height16), Image.LANCZOS)
                    img_tensor = torch.from_numpy(np.array(base_resized).astype(np.float32) / 255.0).unsqueeze(0)
                    with torch.inference_mode(): latent_image = VAEEncode_node.encode(run_vae, img_tensor)[0]
                    mask = _inpaint_mask_np.copy()
                    if "保留" in w_inpaint_mask_mode.value: mask = 1.0 - mask
                    m_pil = Image.fromarray((mask * 255).astype(np.uint8)).resize((width16, height16), Image.LANCZOS)
                    if w_inpaint_mask_blur.value > 0: m_pil = m_pil.filter(ImageFilter.GaussianBlur(radius=w_inpaint_mask_blur.value))
                    m_tensor = torch.from_numpy(np.array(m_pil.resize((width16//8, height16//8), Image.BILINEAR), dtype=np.float32)/255.0).unsqueeze(0)
                    with torch.inference_mode(): latent_image = SetLatentNoiseMask_node.set_mask(latent_image, m_tensor)[0]
                else:
                    latent_image = EmptyLatentImage.generate(width16, height16, batch_size=1)[0]

                print(f"  ⏳ 基础采样中 (denoise={cur_denoise})...")
                # 💥 开启防刺客结界
                with _vram_panic_mode(run_model, bool(active_loras)):
                    with torch.inference_mode():
                        samples = KSampler.sample(run_model, real_seed, steps, cfg, sampler, sched, positive, negative, latent_image, denoise=cur_denoise)[0]
                del latent_image
                print(f"  ✅ 采样完毕，准备收尾清理...")

                # ══════════════════════════════════════════════════════
                # 路由分支：各放大策略
                # ══════════════════════════════════════════════════════

                # ── 1. 无放大直出 ──────────────────────────────────────
                if "无" in upscale_mode:
                    if on_demand:
                        _force_unload_patcher(run_model); run_model = None
                        _engine._destroy_transformer()
                        gc.collect(); gc.collect(); _soft_vram_cleanup()
                        print(f"  🔪 Trans 已彻底剥离并销毁，为 VAE 解码腾出绝对空间[{_format_vram()}]")

                    print(f"  ⏳ VAE 解码中...")
                    with torch.inference_mode():
                        decoded_cpu_batch = (VAEDecodeTiled.decode(run_vae, samples, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, samples)[0]).cpu()
                    final_img = Image.fromarray(np.clip(decoded_cpu_batch[0].numpy() * 255.0, 0, 255).astype(np.uint8))

                # ── 2. Hires.fix ───────────────────────────────────────
                elif "Hires" in upscale_mode:
                    target_w = _round_to_16(int(width16 * scale_val))
                    target_h = _round_to_16(int(height16 * scale_val))
                    upscaler_key = w_hires_upscaler.value
                    h_steps   = w_hires_2nd_steps.value if w_hires_2nd_steps.value > 0 else steps
                    h_cfg     = w_hires_2nd_cfg.value   if w_hires_2nd_cfg.value   > 0 else cfg
                    h_sampler = w_sampler.value if "跟随" in w_hires_2nd_sampler.value  else w_hires_2nd_sampler.value
                    h_sched   = sched           if "跟随" in w_hires_2nd_scheduler.value else w_hires_2nd_scheduler.value
                    h_denoise = denoise_val
                    noise_aug = w_hires_noise_aug.value
                    sharpen_a = w_hires_latent_sharpen.value
                    blend_a   = w_hires_blend.value

                    if "像素空间" in upscaler_key:
                        model_name = "4x-UltraSharp.pth" if "UltraSharp" in upscaler_key else "RealESRGAN_x4plus.pth"
                        print(f"  🔍 Hires.fix [像素空间] {model_name} x{scale_val} → {target_w}×{target_h}")

                        if on_demand:
                            # 提前踢掉 Trans 腾空间给像素放大
                            _force_unload_patcher(run_model); run_model = None
                            _engine._destroy_transformer()
                            gc.collect(); gc.collect(); _soft_vram_cleanup()

                        print(f"  📥 阶段1: 解码基础图...")
                        with torch.inference_mode():
                            base_pixels = (VAEDecodeTiled.decode(run_vae, samples, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, samples)[0])
                        original_latent_for_blend = None
                        if blend_a > 0: original_latent_for_blend = samples["samples"].clone()
                        del samples

                        print(f"  🔬 阶段2: 加载放大模型 {model_name}...")
                        try:
                            from comfy_extras.nodes_upscale_model import UpscaleModelLoader, ImageUpscaleWithModel
                            from nodes import ImageScale
                            _up_model = UpscaleModelLoader().load_model(model_name)[0]
                            with torch.inference_mode():
                                upscaled_px = ImageUpscaleWithModel().upscale(_up_model, base_pixels)[0]
                                if upscaled_px.shape[2] != target_h or upscaled_px.shape[3] != target_w:
                                    upscaled_px = ImageScale().upscale(upscaled_px, "bicubic", target_w, target_h, "center")[0]
                            del base_pixels, _up_model; _flush_after_madvise()
                        except Exception as e:
                            print(f"  ⚠️ 模型放大失败, 回退双三次: {e}")
                            from nodes import ImageScaleBy
                            with torch.inference_mode():
                                upscaled_px = ImageScaleBy().upscale(base_pixels, "bicubic", scale_val)[0]
                            del base_pixels; _flush_after_madvise()

                        print(f"  📤 阶段3: 重新编码到潜空间...")
                        with torch.inference_mode():
                            upscaled_latent = NODE_CLASS_MAPPINGS["VAEEncode"]().encode(run_vae, upscaled_px)[0]
                        del upscaled_px; _flush_after_madvise()

                        if on_demand:
                            print(f"  📥[Phase-2 · 阶段二重载] 加载 Transformer: {_engine._unet_name}")
                            _engine._stream_load_transformer()
                            run_model = getattr(_engine, 'model')
                            if active_loras:
                                for l_name, s_m, s_c in active_loras:
                                    run_model = _apply_unet_lora(run_model, l_name, s_m)

                    else:
                        method = "bislerp" if "bislerp" in upscaler_key else "bicubic"
                        print(f"  🔍 Hires.fix[潜空间] {method} x{scale_val} → {target_w}×{target_h}")
                        LatentUpscale_node = NODE_CLASS_MAPPINGS.get("LatentUpscale", NODE_CLASS_MAPPINGS.get("LatentUpscaleBy"))()
                        original_latent_for_blend = None
                        if blend_a > 0: original_latent_for_blend = samples["samples"].clone()
                        with torch.inference_mode():
                            if "LatentUpscale" in NODE_CLASS_MAPPINGS and not isinstance(LatentUpscale_node, NODE_CLASS_MAPPINGS.get("LatentUpscaleBy", type(None))):
                                upscaled_latent = LatentUpscale_node.upscale(samples, method, target_w, target_h, "center")[0]
                            else:
                                upscaled_latent = NODE_CLASS_MAPPINGS["LatentUpscaleBy"]().upscale(samples, method, scale_val)[0]
                        del samples

                    with torch.inference_mode():
                        lat_s = upscaled_latent["samples"]
                        if sharpen_a > 0:
                            kernel_size = 3; padding = kernel_size // 2; channels = lat_s.shape[1]
                            avg_kernel = torch.ones(channels, 1, kernel_size, kernel_size, device=lat_s.device, dtype=lat_s.dtype) / (kernel_size ** 2)
                            low_pass = F.conv2d(lat_s, avg_kernel, padding=padding, groups=channels)
                            lat_s = lat_s + sharpen_a * (lat_s - low_pass)
                            upscaled_latent["samples"] = lat_s
                            print(f"  🔪 潜空间锐化: α={sharpen_a}")
                        if noise_aug > 0:
                            upscaled_latent["samples"] = lat_s + torch.randn_like(lat_s) * noise_aug
                            print(f"  📢 噪声注入: σ={noise_aug}")
                        if blend_a > 0 and original_latent_for_blend is not None:
                            LatentUpscale2 = NODE_CLASS_MAPPINGS.get("LatentUpscale", NODE_CLASS_MAPPINGS.get("LatentUpscaleBy"))()
                            ref_latent = LatentUpscale2.upscale({"samples": original_latent_for_blend}, "bicubic", target_w, target_h, "center")[0]
                            upscaled_latent["samples"] = (1 - blend_a) * upscaled_latent["samples"] + blend_a * ref_latent["samples"]
                            del original_latent_for_blend, ref_latent

                    print(f"  ⏳ 二阶段精修 (steps={h_steps}, cfg={h_cfg}, sampler={h_sampler}, scheduler={h_sched}, denoise={h_denoise})...")
                    with _vram_panic_mode(run_model, bool(active_loras)):
                        with torch.inference_mode():
                            samples = KSampler.sample(run_model, real_seed + 1, h_steps, h_cfg, h_sampler, h_sched, positive, negative, upscaled_latent, denoise=h_denoise)[0]
                    del upscaled_latent

                    if on_demand:
                        _force_unload_patcher(run_model); run_model = None
                        _engine._destroy_transformer()
                        gc.collect(); gc.collect(); _soft_vram_cleanup()
                        print(f"  🔪 Trans 已彻底销毁，VAE 独占解码[{_format_vram()}]")

                    print(f"  ⏳ VAE 解码中...")
                    with torch.inference_mode():
                        decoded_cpu_batch = (VAEDecodeTiled.decode(run_vae, samples, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, samples)[0]).cpu()
                    final_img = Image.fromarray(np.clip(decoded_cpu_batch[0].numpy() * 255.0, 0, 255).astype(np.uint8))

                # ── 4. Refiner ─────────────────────────────────────────
                elif "Refiner" in upscale_mode:
                    print(f"  🔁 Refiner 模式: 原生比例 Latent 直接精修...")
                    h_steps   = w_hires_2nd_steps.value if w_hires_2nd_steps.value > 0 else steps
                    h_cfg     = w_hires_2nd_cfg.value   if w_hires_2nd_cfg.value   > 0 else cfg
                    h_sampler = w_sampler.value if "跟随" in w_hires_2nd_sampler.value  else w_hires_2nd_sampler.value
                    h_sched   = sched           if "跟随" in w_hires_2nd_scheduler.value else w_hires_2nd_scheduler.value
                    h_denoise = denoise_val
                    noise_aug = w_hires_noise_aug.value
                    sharpen_a = w_hires_latent_sharpen.value

                    with torch.inference_mode():
                        lat_s = samples["samples"]
                        if sharpen_a > 0:
                            kernel_size = 3; padding = kernel_size // 2; channels = lat_s.shape[1]
                            avg_kernel = torch.ones(channels, 1, kernel_size, kernel_size, device=lat_s.device, dtype=lat_s.dtype) / (kernel_size ** 2)
                            low_pass = F.conv2d(lat_s, avg_kernel, padding=padding, groups=channels)
                            lat_s = lat_s + sharpen_a * (lat_s - low_pass)
                            samples["samples"] = lat_s
                        if noise_aug > 0:
                            samples["samples"] = samples["samples"] + torch.randn_like(lat_s) * noise_aug

                    print(f"  ⏳ Refiner 二次采样 (steps={h_steps}, cfg={h_cfg}, sampler={h_sampler}, scheduler={h_sched}, denoise={h_denoise})...")
                    with _vram_panic_mode(run_model, bool(active_loras)):
                        with torch.inference_mode():
                            samples = KSampler.sample(run_model, real_seed + 1, h_steps, h_cfg, h_sampler, h_sched, positive, negative, samples, denoise=h_denoise)[0]

                    if on_demand:
                        _force_unload_patcher(run_model); run_model = None
                        _engine._destroy_transformer()
                        gc.collect(); gc.collect(); _soft_vram_cleanup()
                        print(f"  🔪 Trans 已彻底销毁，VAE 独占解码[{_format_vram()}]")

                    print(f"  ⏳ VAE 解码中...")
                    with torch.inference_mode():
                        decoded_cpu_batch = (VAEDecodeTiled.decode(run_vae, samples, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, samples)[0]).cpu()
                    final_img = Image.fromarray(np.clip(decoded_cpu_batch[0].numpy() * 255.0, 0, 255).astype(np.uint8))

                # ── 3. Out-of-Core 切片放大 ────────────────────────────
                elif "3." in upscale_mode:
                    t_usdu = time.time()

                    if on_demand:
                        _force_unload_patcher(run_model); run_model = None
                        _engine._destroy_transformer()
                        gc.collect(); gc.collect(); _soft_vram_cleanup()
                        print(f"  🔪 Trans 已彻底销毁，基础图 VAE 独占解码[{_format_vram()}]")

                    print(f"  🧩 阶段 1: 提取基础像素图...")
                    with torch.inference_mode():
                        base_image_tensor = VAEDecodeTiled.decode(run_vae, samples, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, samples)[0]
                    del samples; _flush_after_madvise()

                    print(f"  🧩 阶段 2: 放大模型处理...")
                    target_w, target_h = _round_to_16(int(width16 * scale_val)), _round_to_16(int(height16 * scale_val))
                    try:
                        from comfy_extras.nodes_upscale_model import UpscaleModelLoader, ImageUpscaleWithModel
                        from nodes import ImageScale
                        _ooc_model_name = "RealESRGAN_x4plus.pth" if "RealESRGAN" in w_ooc_upscaler.value else "4x-UltraSharp.pth"
                        print(f"  🔬 OOC 放大模型: {_ooc_model_name}")
                        upscale_model = UpscaleModelLoader().load_model(_ooc_model_name)[0]
                        with torch.inference_mode():
                            upscaled_4x = ImageUpscaleWithModel().upscale(upscale_model, base_image_tensor)[0]
                            target_tensor = ImageScale().upscale(upscaled_4x, "bicubic", target_w, target_h, "center")[0]
                        del upscaled_4x, upscale_model
                    except Exception as e:
                        print(f"  ⚠️ 模型放大失败，回退双三次: {e}")
                        from nodes import ImageScaleBy
                        target_tensor = ImageScaleBy().upscale(base_image_tensor, "bicubic", scale_val)[0]
                    finally:
                        if 'base_image_tensor' in locals() and base_image_tensor is not None: del base_image_tensor
                        _flush_after_madvise()

                    target_pil = Image.fromarray(np.clip(target_tensor[0].cpu().numpy() * 255.0, 0, 255).astype(np.uint8))
                    del target_tensor; gc.collect()

                    print(f"  🧩 阶段 3: Out-of-Core 硬盘切片独立重绘...")
                    os.makedirs("/tmp/zimg_tiles", exist_ok=True)
                    grid_map = {"4块 (2x2)": 2, "9块 (3x3)": 3, "16块 (4x4)": 4}
                    grid_n = grid_map.get(w_ooc_grid.value, 2)
                    overlap_px = 64
                    tile_w = _ceil_to_16((target_w + overlap_px * (grid_n - 1)) / grid_n)
                    tile_h = _ceil_to_16((target_h + overlap_px * (grid_n - 1)) / grid_n)
                    tile_w, tile_h = max(256, tile_w), max(256, tile_h)
                    coords = get_optimal_tile_coords(target_w, target_h, tile_w, tile_h, overlap=overlap_px)
                    print(f"  🔪 切块阵列: {grid_n}x{grid_n} → 单块 {tile_w}×{tile_h} (实际 {len(coords)} 块)")

                    out_tile_paths =[]
                    VAEEncode_node = NODE_CLASS_MAPPINGS["VAEEncode"]()

                    if on_demand:
                        print(f"  📥[Phase-2 · OOC重载] 加载 Transformer: {_engine._unet_name}")
                        _engine._stream_load_transformer()
                        run_model = getattr(_engine, 'model')
                        if active_loras:
                            for l_name, s_m, s_c in active_loras:
                                run_model = _apply_unet_lora(run_model, l_name, s_m)

                    for idx, box in enumerate(coords):
                        t_start = time.time()
                        print(f"    ➡️ 重绘切块[{idx+1}/{len(coords)}] 坐标: {box}...")
                        tile_crop = target_pil.crop(box)
                        tile_tensor = torch.from_numpy(np.array(tile_crop).astype(np.float32) / 255.0).unsqueeze(0)
                        with torch.inference_mode():
                            tile_latent = VAEEncode_node.encode(run_vae, tile_tensor)[0]
                            with _vram_panic_mode(run_model, bool(active_loras)):
                                tile_sampled = KSampler.sample(run_model, real_seed + idx, steps, cfg, sampler, sched, positive, negative, tile_latent, denoise=denoise_val)[0]
                            del tile_latent
                            tile_decoded = (VAEDecodeTiled.decode(run_vae, tile_sampled, tile_size)[0] if use_tiled else VAEDecode.decode(run_vae, tile_sampled)[0]).cpu()
                            del tile_sampled
                        out_tile_pil = Image.fromarray(np.clip(tile_decoded[0].numpy() * 255.0, 0, 255).astype(np.uint8))
                        t_path = f"/tmp/zimg_tiles/out_{idx}.png"
                        out_tile_pil.save(t_path)
                        out_tile_paths.append(t_path)
                        del tile_tensor, tile_decoded, out_tile_pil, tile_crop
                        gc.collect(); _flush_after_madvise()

                    del target_pil; _flush_after_madvise()

                    if on_demand:
                        _force_unload_patcher(run_model); run_model = None
                        _engine._destroy_transformer()
                        gc.collect(); _soft_vram_cleanup()
                        print(f"  🔪 Trans 已彻底销毁，切块重绘完成[{_format_vram()}]")

                    print(f"  🧵 阶段 4: 像素级羽化无缝拼接...")
                    final_canvas = Image.new("RGB", (target_w, target_h))
                    for idx, box in enumerate(coords):
                        with Image.open(out_tile_paths[idx]).convert("RGB") as t_pil:
                            final_canvas = blend_images(final_canvas, t_pil, box, overlap=64)
                    print(f"  ✅ 拼接完成！耗时 {time.time()-t_usdu:.1f}s")
                    final_img = final_canvas

                # ── Meta 记录 ──────────────────────────────────────────
                meta = {
                    "prompt": prompt_str, "negative_prompt": neg_str, "seed": real_seed,
                    "steps": steps, "cfg_scale": cfg, "width": width16, "height": height16,
                    "sampler_name": sampler, "scheduler": sched, "vae_tiled": use_tiled,
                    "upscale_mode": upscale_mode, "upscale_scale": scale_val,
                    "upscale_denoise": denoise_val, "upscale_grid": w_ooc_grid.value,
                    "ooc_upscaler": w_ooc_upscaler.value
                }
                if w_svh_enable.value: meta["seed_variance_enhancer"] = svh_meta
                if "Hires" in upscale_mode:
                    meta.update({
                        "hires_upscaler": w_hires_upscaler.value,
                        "hires_2nd_steps": w_hires_2nd_steps.value, "hires_2nd_cfg": w_hires_2nd_cfg.value,
                        "hires_2nd_sampler": w_hires_2nd_sampler.value, "hires_2nd_scheduler": w_hires_2nd_scheduler.value,
                        "hires_noise_aug": w_hires_noise_aug.value, "hires_latent_sharpen": w_hires_latent_sharpen.value,
                        "hires_blend": w_hires_blend.value
                    })
                if "Refiner" in upscale_mode:
                    meta.update({
                        "hires_2nd_steps": w_hires_2nd_steps.value, "hires_2nd_cfg": w_hires_2nd_cfg.value,
                        "hires_2nd_sampler": w_hires_2nd_sampler.value, "hires_2nd_scheduler": w_hires_2nd_scheduler.value,
                        "hires_noise_aug": w_hires_noise_aug.value, "hires_latent_sharpen": w_hires_latent_sharpen.value
                    })

                prefix = "ZImg_Inpaint" if is_inp else "ZImg"
                save_path = os.path.join(out_dir, f"{prefix}_{int(time.time())}_{i}.png")
                pnginfo = PngImagePlugin.PngInfo()
                pnginfo.add_itxt("parameters", json.dumps(meta, ensure_ascii=False), lang="", tkey="")
                final_img.save(save_path, format="PNG", pnginfo=pnginfo)
                print(f"  💾 {save_path}")
                display(final_img)

            except Exception as e:
                print(f"\n❌ 第{i+1}张生成出错: {e}")
                import traceback; traceback.print_exc()
            finally:
                if on_demand:
                    _force_unload_patcher(run_model)
                    try: _engine._destroy_transformer()
                    except: pass
                else:
                    _force_unload_patcher(run_model)

                _l = locals()
                for v in['run_model', 'run_clip', 'run_vae', 'positive', 'negative', 'latent_image',
                           'samples', 'decoded_cpu_batch', 'final_img', 'base_image_tensor',
                           'upscaled_4x', 'target_tensor', 'target_pil', 'tile_tensor', 'tile_decoded']:
                    if v in _l and _l[v] is not None:
                        try: del _l[v]
                        except: pass
                gc.collect(); gc.collect(); _soft_vram_cleanup()
                print(f"  🏁 {_format_vram()} | {_format_ram()}")

btn_run.on_click(run_generation)
btn_clear.on_click(lambda _: out.clear_output())
btn_full_unload.on_click(_on_full_unload)

# ==========================================
# 7. UI 布局拼接与最终渲染
# ==========================================
sec_prompt = make_section("📝 提示词与重绘预设", _col(
    widgets.HTML('<div class="zimg-label">正向提示词</div>'), prompt_box,
    widgets.HTML('<div class="zimg-label" style="margin-top:6px;">负面提示词</div>'), neg_box,
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    _row(w_png_meta_path, btn_load_meta), meta_status,
    w_inpaint_enable, inpaint_box
))
sec_params = make_section("⚙️ 尺寸与生成参数", _col(
    w_ratio, _row(w_width, w_height), w_size_info,
    widgets.HTML('<hr style="border-color:#334155;margin:8px 0;">'),
    _row(w_steps, w_cfg), _row(w_seed, w_batch), _row(w_sampler, w_scheduler)
))
sec_upscale = make_section("🔍 放大策略 (硬盘 OOC 无限大图)", _col(
    w_upscale_mode,
    _row(w_upscale_scale, w_upscale_denoise, w_ooc_grid),
    _row(w_ooc_upscaler),
    hires_extra,
    widgets.HTML('<div class="zimg-info">💡 智能 OOC 阵列：程序会自动测算贴合您选择块数的最佳尺寸，绝不多切一块，并保证无缝衔接。</div>')
))
sec_lora = make_section("🎭 三槽位 LoRA", _col(
    _row(btn_refresh_lora, widgets.HTML("<span class='zimg-info'>存放目录: /content/ComfyUI/models/loras/</span>")),
    _row(lora_dd_1, lora_sm_1, lora_sc_1),
    _row(lora_dd_2, lora_sm_2, lora_sc_2),
    _row(lora_dd_3, lora_sm_3, lora_sc_3),
    widgets.HTML('<div class="zimg-info">💡 串行模式下 LoRA 自动相位分离：CLIP权重→Phase-1(TE独占)，UNet权重→Phase-2(Trans独占)，两者永不同台。</div>')
))
sec_adv = make_section("🔬 高级选项 & 保存", _col(
    widgets.HTML('<div class="zimg-label">基础图 VAE 解码</div>'), w_vae_tiled, w_tile_size,
    widgets.HTML('<hr style="border-color:#334155;margin:10px 0;">'),
    widgets.HTML('<div class="zimg-label">出图多样性 (SVE)</div>'), w_svh_enable, sve_controls,
    widgets.HTML('<hr style="border-color:#334155;margin:10px 0;">'),
    w_save_custom, w_custom_dir
))

ui_buttons = widgets.HBox([btn_run, btn_clear, btn_full_unload], layout=widgets.Layout(justify_content="center", margin="16px 0", gap="12px", align_items="center"))

main_ui = widgets.VBox([
    header_html,
    allocator_html,
    trans_mode_html,
    sec_prompt, sec_params, sec_upscale, sec_lora, sec_adv,
    ui_buttons,
    widgets.HTML('<div class="zimg-section-title" style="margin-top:10px;">📺 图像输出台</div>'),
    out
], layout=widgets.Layout(width="820px", padding="16px", border_radius="16px", background="#0f172a"))

display(main_ui)

*使用说明：*

1，两种模型加载模式，“电焊模式”，z image turbo 双Q_8(Trans+TE)+Lora可直出2048x2048不爆显存；“流式加载模式”，z image turbo 双F16(Trans+TE)，可直出2048x2048不爆显存（F16_TE意义不大，建议Q_8_TE)，但无法加载Lora，否则显存必爆。

2，要冲击z image turbo F16_Trans+Lora，可先进行Trans+Lora融合，生成已经融合了Lora的“专属F16_GGUF”;再用模型管理器流式加载融合GGUF，效果完全等同于加载多个Lora。

3，融合GGUF结构完全改变，使用时无需再加载Lora，也无需触发词。

4，GGUF可融合多个Lora，但真人Lora只能融合一个，否则生成的是“融合脸”。

5，由于仅融合了UNet，对于风格Lora，或含风格的Lora，仍需在出图时，加载该Lora，强度设为“0”，clip设为正常需要的强度。